# 02j — Budget-response sweep: WHY five languages ignore the phoneme budget

At checkpoint 3801 six languages follow the budget and five do not. A slope alone cannot
say which of four different defects that is, and they need four different fixes.

This session sweeps the budget **wide** (0.4x to 2.0x, against the old 0.6x–1.4x) on a
fixed sentence and keeps **every raw point**, so the shape of the response is readable
rather than just its slope.

Two things are different from every previous session in this phase.

**It is zero-referenced.** The old probe pooled raw lengths across sentences, which let
sentence-length variance leak into the fit: a model that ignores the budget entirely scored
0.60–0.83 depending on the language, not 0. Six of eleven languages were at or below their
own floor. This run reports the within-sentence normalised slope, where 0 means ignored.

**It pays for itself.** Every generation here is exactly measurable — the phoneme counter
is exact, and the semantic gate is the one production uses. So any generation that
verifiably hit its budget without losing its meaning is a training row for the corrective
fine-tune, harvested from work that had to happen anyway. Decision-critical sweep first,
harvest second, hard wall-clock stop throughout.


## 1. Environment (reused verbatim from 02i — six attempts to get right)

In [ ]:
# Kaggle secrets. This runs BEFORE anything expensive, on purpose.
#
# A secret lives on the account but must be ticked for EACH notebook individually
# (Add-ons -> Secrets). An unattached secret does not report as missing: it raises
# "Connection error trying to communicate with service", which reads like a transient
# outage and invites a retry loop that can never succeed. Sessions 02i and 02j both hit
# this. 02i treated it as a warning and burned ~12 GPU-hours with no live channel.
import os

_WANTED = {
    # name             fatal   why
    "WANDB_API_KEY":  (True,  "Kaggle publishes a notebook log only when the session ENDS, "
                              "so W&B is the only channel that reports while this runs. A "
                              "blind multi-hour session is not cheaper than no session."),
    "HF_TOKEN":       (False, "Needed for gated repos such as meta-llama/*. Unsloth "
                              "resolves to its own non-gated 4-bit mirror, so a run can "
                              "survive without this - but it will fail late and "
                              "confusingly if it ever does need it."),
}

_client = None
_status = {}
for _name, (_fatal, _why) in _WANTED.items():
    if os.environ.get(_name):
        _status[_name] = "already in environment"
        continue
    try:
        if _client is None:
            from kaggle_secrets import UserSecretsClient
            _client = UserSecretsClient()
        os.environ[_name] = _client.get_secret(_name)
        _status[_name] = "loaded from Kaggle secrets"
    except Exception as _e:
        _status[_name] = f"UNAVAILABLE ({type(_e).__name__}: {_e})"
        if _fatal:
            raise RuntimeError(
                "{} could not be loaded: {}\n"
                "  Open this notebook -> Add-ons -> Secrets -> tick {}.\n"
                "  Note: an UNATTACHED secret reports as a connection error, so this is "
                "almost always an attachment problem rather than an outage.\n"
                "  Refusing to start. {}".format(_name, _e, _name, _why)) from _e
        print("WARNING: {} {} - {}".format(_name, _status[_name], _why))

# Aliases the HF libraries actually read.
if os.environ.get("HF_TOKEN"):
    os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", os.environ["HF_TOKEN"])

os.environ.setdefault("WANDB_ENTITY", "nktthegreat-soccernet")
os.environ.setdefault("WANDB_PROJECT", "indic-dubbing-v3")

# Never print a value. A W&B key is 40 hex characters; anything else is the wrong string,
# and catching that here beats catching it at wandb.init after the model is loaded.
for _name, _s in _status.items():
    _v = os.environ.get(_name) or ""
    _len = "{} chars".format(len(_v)) if _v else "absent"
    _flag = ""
    if _name == "WANDB_API_KEY" and _v and len(_v) != 40:
        _flag = "  <-- expected 40, check the secret"
    print("{:<16} {:<28} {}{}".format(_name, _s, _len, _flag))


In [ ]:
# Record the platform's torch BEFORE installing anything. Kaggle's torch is built to match
# the driver and the T4's compute capability (sm_75). `pip install unsloth` will happily
# pull a newer torch whose binaries have no sm_75 kernels, and the failure is
# `CUDA error: no kernel image is available for execution on the device` — which appears
# only when the first kernel actually launches, long after the install "succeeded".
# requirements.txt warns about this in capitals; the second attempt at this notebook
# ignored it and lost a GPU session.
import torch
TORCH_BEFORE = torch.__version__
print("platform torch:", TORCH_BEFORE, "| device:", torch.cuda.get_device_name(0))
print("compute capability:", torch.cuda.get_device_capability(0))


In [ ]:
# Plain install, WITH dependencies. An earlier attempt used `--no-deps` to stop pip
# replacing torch — but torch was never the problem (it was a Tesla P100 assignment, see
# machine_shape), and `--no-deps` then left torchao pinned at 0.10.0 while transformers
# requires >0.16, which failed inside the eval subprocess. Solving a hypothetical risk
# created a real one. This is what the original 02_llm_finetune notebook used and what is
# known to work on a T4.
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y espeak-ng > /dev/null 2>&1
!pip -q install unsloth 2>&1 | tail -2
!pip -q install phonemizer indic-nlp-library sacrebleu sentence-transformers wandb 2>&1 | tail -2


In [ ]:
# THE PREFLIGHT THAT MATTERS. Not "did pip succeed", not "is CUDA available" — both were
# true in the run that died. Three real checks, each of which fails in seconds:
#   1. torch is still the platform build (sm_75 kernels present)
#   2. a kernel actually launches on this device
#   3. the heavy import really resolves, which is where --no-deps gaps surface
import subprocess, torch

print(subprocess.run(["espeak-ng", "--version"], capture_output=True, text=True).stdout.strip())
print(f"torch: {torch.__version__} (was {TORCH_BEFORE})")
if torch.__version__ != TORCH_BEFORE:
    print(f"NOTE: torch changed {TORCH_BEFORE} -> {torch.__version__}. Fine on a T4 (sm_75);"
          f" only sm_60 (P100) is unsupported by current builds.")
cap = torch.cuda.get_device_capability(0)
assert cap >= (7, 0), (
    f"GPU is compute capability {cap} ({torch.cuda.get_device_name(0)}). Current torch "
    f"builds ship sm_70+ only, so nothing will execute. Push with "
    f"machine_shape=NvidiaTeslaT4.")

x = torch.randn(64, 64, device="cuda")
assert torch.isfinite(x @ x).all().item(), "GPU matmul did not produce finite values"
torch.cuda.synchronize()
print("GPU kernel launch: OK")

from unsloth import FastLanguageModel          # the import that actually exercises the deps
import phonemizer, indicnlp, sacrebleu, sentence_transformers
print("imports OK: unsloth phonemizer indicnlp sacrebleu sentence_transformers")


## 2. Embedded modules

Read from `pipeline_v3/` at build time, not hand-copied, so the notebook cannot drift from the repo.

In [ ]:
import os, sys
for d in ('common', 'translation', 'evaluation', 'tools'):
    os.makedirs(f'/kaggle/working/pipeline_v3/{d}', exist_ok=True)
sys.path.insert(0, '/kaggle/working/pipeline_v3')
os.chdir('/kaggle/working/pipeline_v3')
print(os.getcwd())


In [ ]:
%%writefile /kaggle/working/pipeline_v3/common/__init__.py


In [ ]:
%%writefile /kaggle/working/pipeline_v3/common/languages.py
"""
common/languages.py
--------------------
Single source of truth for language metadata used across pipeline_v3.

Every other module (dataset_generator, duration_predictor, isochrony_translation_v3,
train_translation_llm, data_augmentation, train_tts) imports from here instead of
hard-coding its own language table. This avoids the classic bug where "hi" is spelled
one way in one file and another way in another file.

The 11 languages below are the intersection of:
  - Samanantar's 11 Indic targets (as, bn, gu, hi, kn, ml, mr, or, pa, ta, te)
  - IndicF5's 11 supported languages (same set)
  - espeak-ng's Indic voice coverage (verified: all 11 present as of espeak-ng 1.51)

NOTE ON EXPANSION RATIOS: The `heuristic_expansion_ratio` and `heuristic_phonemes_per_sec`
values below are *cold-start defaults*, carried forward from/consistent with the existing
V2 pipeline's phoneme_counter.py approach (e.g. Hindi=1.30, Tamil=1.35). They are
deliberately approximate. The entire point of V3 is to replace these hard-coded numbers
with the learned DurationPredictor (see translation/duration_predictor.py) once you have
trained it on real forced-aligned speech. Treat this table as the fallback that is used
(a) before you've trained anything, and (b) as a sanity-check ceiling/floor on predicted
durations even after training.
"""

from dataclasses import dataclass


@dataclass(frozen=True)
class LanguageInfo:
    name: str                          # Human-readable display name
    iso_code: str                      # 2-letter ISO 639-1 code, used by Samanantar & IndicF5
    espeak_code: str                   # espeak-ng voice code (for phonemizer backend="espeak")
    indicf5_code: str                  # Code IndicF5 expects (matches iso_code for all 11)
    script: str                        # Unicode script name, for sanity checks / logging
    heuristic_expansion_ratio: float   # Indic-text-length / English-text-length, rough prior
    heuristic_phonemes_per_sec: float  # Cold-start speaking rate for DurationPredictor fallback


# fmt: off
LANGUAGES: dict[str, LanguageInfo] = {
    "hi": LanguageInfo("Hindi",      "hi", "hi", "hi", "Devanagari", 1.30, 13.5),
    "bn": LanguageInfo("Bengali",    "bn", "bn", "bn", "Bengali",    1.28, 13.0),
    "mr": LanguageInfo("Marathi",    "mr", "mr", "mr", "Devanagari", 1.27, 13.2),
    "gu": LanguageInfo("Gujarati",   "gu", "gu", "gu", "Gujarati",   1.25, 13.0),
    "pa": LanguageInfo("Punjabi",    "pa", "pa", "pa", "Gurmukhi",   1.22, 12.8),
    "ta": LanguageInfo("Tamil",      "ta", "ta", "ta", "Tamil",      1.35, 12.5),
    "te": LanguageInfo("Telugu",     "te", "te", "te", "Telugu",     1.33, 12.6),
    "kn": LanguageInfo("Kannada",    "kn", "kn", "kn", "Kannada",    1.30, 12.7),
    "ml": LanguageInfo("Malayalam",  "ml", "ml", "ml", "Malayalam",  1.38, 12.3),
    "or": LanguageInfo("Odia",       "or", "or", "or", "Odia",       1.26, 13.0),
    "as": LanguageInfo("Assamese",   "as", "as", "as", "Bengali",    1.27, 12.9),
}
# fmt: on

# Samanantar's HF dataset config names are lowercase 2-letter codes identical to iso_code,
# EXCEPT the source side, which is always English.
SAMANANTAR_SOURCE_LANG = "en"
SAMANANTAR_HF_PATH = "ai4bharat/samanantar"

# ai4bharat/Kathbath directory structure uses full language *names* (lowercase), not codes.
KATHBATH_DIR_NAMES: dict[str, str] = {
    "hi": "hindi", "bn": "bengali", "mr": "marathi", "gu": "gujarati",
    "pa": "punjabi", "ta": "tamil", "te": "telugu", "kn": "kannada",
    "ml": "malayalam", "or": "odia", "as": "assamese",
    # Kathbath additionally has Sanskrit/Urdu/Nepali/Chhattisgarhi splits not covered here
    # because they fall outside the 11-language Samanantar/IndicF5 intersection.
}
KATHBATH_HF_PATH = "ai4bharat/Kathbath"

# ai4bharat/Rasa is the (much closer to IndicF5's own training mix) TTS-quality speech
# dataset: studio recordings, 13 languages, ~500+ hours, MIT-tagged on HF. Prefer this over
# Kathbath (which is ASR-oriented, phone/field recordings) for anything TTS-related, and
# fall back to Kathbath only if you need more raw hours than Rasa provides for a language.
RASA_HF_PATH = "ai4bharat/Rasa"

# ai4bharat/BPCC (Bharat Parallel Corpus Collection) - the successor to Samanantar:
# ~230M pairs, 22 languages, and (unlike standalone Samanantar's ambiguous HF license
# tag) an EXPLICIT license table on its dataset card: all MINED corpora - including
# Samanantar (19.4M) and Samanantar++ (121.6M) - are CC0; the human-annotated seed
# subsets (BPCC-H-Wiki/Daily) are CC-BY-4.0. This resolves the Samanantar license
# ambiguity flagged in dataset_generator.py for commercial use. GATED: accept the
# conditions on huggingface.co/datasets/ai4bharat/BPCC with your HF account first.
#
# STRUCTURE (verified 2026-07-13 by browsing the gated repo directly, including
# AI4Bharat's own compile.py in the repo root, which generated these files):
# BPCC is a RAW-FILE repo, not a config-per-language dataset - the HF dataset viewer is
# disabled for it and it has no loading script, so load_dataset("ai4bharat/BPCC",
# <config>) does NOT work. The data lives in per-subset directories of per-language
# TSVs keyed by FLORES-200 code:
#     <subset>/<flores_code>.tsv    e.g.  samanantar_v2/hin_Deva.tsv  (6.9 GB)
# Each TSV is tab-separated WITH a header row and exactly these columns (from
# compile.py: df.to_csv(sep='\t', index=False)):
#     src_lang  tgt_lang  src  tgt      (src = English text, tgt = Indic text)
# All 11 of our languages are present in samanantar_v2/ (plus npi/urd, unused here).
# dataset_generator.py streams these via load_dataset("csv", data_files="hf://...").
# Subsets seen in the repo: samanantar_v2 (default - 28.9GB, the filtered mined set),
# samanantar_v0.3_filtered, nllb_filtered, nllb_seed, bpcc-seed-latest/v1/v2,
# comparable, daily, ilci, massive, wiki. Only samanantar_v2's internal layout was
# inspected; treat other subsets' layouts as unverified until probed.
BPCC_HF_PATH = "ai4bharat/BPCC"
BPCC_SOURCE_FLORES = "eng_Latn"
BPCC_DEFAULT_SUBSET = "samanantar_v2"
BPCC_FLORES_CODES: dict[str, str] = {
    "hi": "hin_Deva", "bn": "ben_Beng", "mr": "mar_Deva", "gu": "guj_Gujr",
    "pa": "pan_Guru", "ta": "tam_Taml", "te": "tel_Telu", "kn": "kan_Knda",
    "ml": "mal_Mlym", "or": "ory_Orya", "as": "asm_Beng",
}

# IndicF5 itself
INDICF5_HF_PATH = "ai4bharat/IndicF5"
INDICF5_SAMPLE_RATE = 24000

# ai4bharat/vits_rasa_13 - fixed-inventory multi-speaker VITS TTS (40.2M params,
# CC-BY-4.0, GATED). Everything below is transcribed 1:1 from the model card's own
# speaker/style tables (verified 2026-07-13 via authenticated access, not assumed).
# IMPORTANT COVERAGE FACT: the model supports 13 languages but NOT Hindi, Gujarati, or
# Odia - three of this pipeline's 11 targets. hi/gu/or must use the IndicF5 backend
# for multi-speaker dubbing (see tts/vits_rasa_tts.py and multi_speaker_dubbing.py).
VITS_RASA_HF_PATH = "ai4bharat/vits_rasa_13"
VITS_RASA_SPEAKERS: dict[str, int] = {
    "ASM_F": 0, "ASM_M": 1, "BEN_F": 2, "BEN_M": 3, "BRX_F": 4, "BRX_M": 5,
    "DOI_F": 6, "DOI_M": 7, "KAN_F": 8, "KAN_M": 9, "MAI_M": 10, "MAL_F": 11,
    "MAR_F": 12, "MAR_M": 13, "NEP_F": 14, "PAN_F": 15, "PAN_M": 16, "SAN_M": 17,
    "TAM_F": 18, "TEL_F": 19,
}
# Style/emotion IDs exactly as published (note the real gaps at 9/11/13 - the model
# card skips those IDs; do not "fix" this by renumbering).
VITS_RASA_STYLES: dict[str, int] = {
    "ALEXA": 0, "ANGER": 1, "BB": 2, "BOOK": 3, "CONV": 4, "DIGI": 5, "DISGUST": 6,
    "FEAR": 7, "HAPPY": 8, "NEWS": 10, "SAD": 12, "SURPRISE": 14, "UMANG": 15, "WIKI": 16,
}
# Per-target-language voice availability, keyed by this pipeline's ISO codes.
# None = that gender doesn't exist in the model (ml/ta/te are female-only).
VITS_RASA_VOICES_BY_LANG: dict[str, dict] = {
    "as": {"F": 0, "M": 1},
    "bn": {"F": 2, "M": 3},
    "kn": {"F": 8, "M": 9},
    "ml": {"F": 11, "M": None},
    "mr": {"F": 12, "M": 13},
    "pa": {"F": 15, "M": 16},
    "ta": {"F": 18, "M": None},
    "te": {"F": 19, "M": None},
}
VITS_RASA_UNSUPPORTED: frozenset = frozenset({"hi", "gu", "or"})

# pyannote speaker diarization (multi-speaker dubbing's segmentation stage). BOTH repos
# are gated on HF - accept terms on each model page with the same HF_TOKEN account:
#   huggingface.co/pyannote/speaker-diarization-3.1
#   huggingface.co/pyannote/segmentation-3.0  (pulled in by the pipeline above)
PYANNOTE_DIARIZATION_HF_PATH = "pyannote/speaker-diarization-3.1"


def get_language(code: str) -> LanguageInfo:
    code = code.lower().strip()
    if code not in LANGUAGES:
        raise ValueError(
            f"Unknown language code '{code}'. Supported: {sorted(LANGUAGES.keys())}"
        )
    return LANGUAGES[code]


def all_codes() -> list[str]:
    return list(LANGUAGES.keys())


def display_name(code: str) -> str:
    return get_language(code).name


In [ ]:
%%writefile /kaggle/working/pipeline_v3/common/phonemes.py
"""
common/phonemes.py
==================
THE canonical grapheme-to-phoneme counter for pipeline_v3. Every module that writes a
phoneme budget into a label, and every module that scores a generation against one, must
import `count_phonemes` from here and from nowhere else.

WHY THIS MODULE EXISTS (read this before changing anything in it)
-----------------------------------------------------------------
The previous implementation lived in `translation/duration_predictor.phonemize_text` and
ended like this::

    except Exception as e:
        logger.debug("Phonemization failed ...; falling back to characters.")
    return [c for c in text if not c.isspace()]

That fallback fired for the **entire** corpus generation run, because espeak-ng (a system
binary, installed separately from the `phonemizer` pip package) was not present in that
Kaggle session. It logged at DEBUG, so nothing surfaced. The result: every `n_phonemes`
label in the base training corpus is a **non-space character count**, verified at 100.0%
over 1,289 locally-held rows with a chars/n_phonemes ratio of exactly 1.000 (min = max).

It got worse. A later session (length augmentation, 2026-07-26) *did* have espeak-ng, so
those rows are labelled in **real phonemes** — 8.4% coincidental agreement with character
counts, ratio spread 0.198-1.500. The corpus therefore carries two mutually incompatible
rulers under one prompt token, `[Target Phonemes: N]`, teaching the model two
contradictory tasks. A model asked to fit N of one unit and N of another cannot reach
slope 1.0 on either; it can only split the difference.

So this module enforces three things the old one did not:

1. **No silent fallback, ever.** A phonemization failure raises `G2PUnavailable` or
   `PhonemizationError`. Label-writing code must never degrade quietly, because a
   degraded label is indistinguishable from a good one downstream. Inference code that
   legitimately needs to survive a bad string catches the exception *explicitly* and
   records the degradation (see "degrade, don't crash" — but degrade *visibly*).

2. **A preflight that proves the output is phonemes, not passthrough.** Checking that
   espeak-ng is importable is not sufficient — the failure mode we actually hit produces
   plausible-looking output. `assert_g2p_available()` phonemizes a canary string in each
   language and asserts the returned symbols are not simply the input's own characters.
   That is the check that would have caught this on day one.

3. **A ruler identifier stamped into every artifact.** `ruler_id()` returns a string like
   ``phonemes:espeak-ng-1.51``. Dataset rows, eval reports, and run manifests all carry
   it, so "which ruler produced this number" is a grep, not a forensic exercise.

WHAT COUNTS AS ONE PHONEME
---------------------------
espeak-ng's IPA output carries symbols that are not sounds. We normalise before counting:

- **Stress marks** (``ˈ`` primary, ``ˌ`` secondary) are suprasegmental — they mark which
  syllable is emphasised, not an additional sound. Stripped.
- **Length marks** (``ː``) modify the preceding vowel's duration and stay attached to it,
  so ``aː`` is one (long) phoneme, not two. Kept attached — which is correct for our
  purpose, since duration is exactly what we are proxying.
- **Language-switch tags** (``(en)``, ``(hi)``) are emitted when espeak detects a foreign
  word — typically English brand names inside Indic text. They are markup. Stripped.
- **Tie bars** (``͡``) join affricates into one segment. Kept attached.

These choices are asymmetric-safe: they can only ever be wrong by a constant per language,
and non-negotiable #3 (same function for labels and scores) means a constant offset
cancels. What must never happen is two *different* normalisations in the same project.
"""

from __future__ import annotations

import functools
import logging
import re
from typing import Iterable, Optional, Sequence

from common.languages import LANGUAGES, get_language

logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------------------
# Ruler identifiers. These are written into datasets and reports; treat them as a stable
# public vocabulary, not as free text.
# ---------------------------------------------------------------------------------------

RULER_PHONEMES = "phonemes:espeak-ng"
RULER_CHARS = "chars:non-space"
RULER_UNKNOWN = "unknown"


class G2PUnavailable(RuntimeError):
    """espeak-ng and/or the `phonemizer` package is missing or non-functional.

    Raised by the preflight and by `phonemize()`. This is deliberately fatal: every
    caller in the label-writing path would otherwise produce a corpus that looks correct
    and is measured in the wrong unit.
    """


class PhonemizationError(RuntimeError):
    """espeak-ng is present and working, but this specific string could not be converted."""


_INSTALL_HINT = (
    "espeak-ng is a SYSTEM package and is NOT installed by `pip install phonemizer`.\n"
    "  Kaggle / Debian / Ubuntu:  apt-get -qq update && apt-get -qq install -y espeak-ng\n"
    "  macOS:                     brew install espeak-ng\n"
    "  Windows:                   winget install --id eSpeak-NG.eSpeak-NG   (then set\n"
    "                             PHONEMIZER_ESPEAK_LIBRARY to the installed libespeak-ng.dll)\n"
    "  Then:                      pip install phonemizer\n"
    "Verify with: python -c \"from common.phonemes import assert_g2p_available;"
    " assert_g2p_available()\""
)

# Suprasegmental / markup symbols that are not themselves sounds.
_STRESS_MARKS = "ˈˌ"          # ˈ ˌ
_LANG_SWITCH_RE = re.compile(r"\([a-z]{2,3}\)")   # (en), (hi), ...
_UNDERTIE = "‿"

# Unicode names several scripts by an older label than the one the language table uses.
_UNICODE_SCRIPT_NAME = {"odia": "ORIYA"}


def _script_token(language_iso_code: str) -> str:
    """The token that appears in unicodedata.name() for this language's script."""
    script = get_language(language_iso_code).script.lower()
    return _UNICODE_SCRIPT_NAME.get(script, script.split()[0].upper())


# ---------------------------------------------------------------------------------------
# Orthographic normalisation (AI4Bharat IndicNLP)
# ---------------------------------------------------------------------------------------

@functools.lru_cache(maxsize=16)
def _normalizer(language_iso_code: str):
    """Per-language IndicNLP normaliser, or None if the library is absent."""
    try:
        from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    except ImportError:
        return None
    return IndicNormalizerFactory().get_normalizer(language_iso_code)


# Set once, the first time normalisation is skipped, so the absence is stated rather than
# inferred from a slightly-off number three stages downstream.
_NORMALIZER_WARNED: set = set()


@functools.lru_cache(maxsize=1)
def _recomposition_map() -> dict:
    """decomposed-sequence -> precomposed-character, for Indic nukta letters.

    Built from `unicodedata` rather than hand-listed, so it cannot drift from the standard
    and needs no maintenance when a script is added.

    WHY IT IS NEEDED. IndicNLP canonicalises *toward the decomposed form*: `য়` U+09DF
    becomes U+09AF + U+09BC (YA + NUKTA). espeak-ng's rule files are evidently written
    against the *precomposed* letters. Measured on this corpus: normalisation rewrites
    50.8% of Assamese rows, almost all of them this substitution, and Assamese
    phonemes-per-character then jumps 1.147 -> 1.604 while Bengali — same script, same
    substitution on 36.3% of its rows — barely moves (1.032 -> 1.020). Assamese and Bengali
    are phonologically close and orthographically shared; a 57% divergence between them is
    not a better measurement, it is espeak's `as` voice failing to parse a sequence its
    `bn` voice handles.

    These characters cannot be recomposed by `unicodedata.normalize("NFC", ...)`, because
    Indic nukta letters are Unicode *composition exclusions* — NFC deliberately leaves them
    decomposed. Hence an explicit map.

    So: take IndicNLP's genuine cleanups (ZWJ/ZWNJ removal, punctuation canonicalisation,
    Malayalam chillu handling) and then put the nukta letters back into the encoding the
    G2P actually recognises.
    """
    import unicodedata
    mapping = {}
    # Devanagari, Bengali, Gurmukhi, Gujarati, Oriya, Tamil, Telugu, Kannada, Malayalam
    for cp in range(0x0900, 0x0E00):
        ch = chr(cp)
        decomp = unicodedata.decomposition(ch)
        if not decomp or decomp.startswith("<"):   # skip compatibility decompositions
            continue
        try:
            seq = "".join(chr(int(p, 16)) for p in decomp.split())
        except ValueError:
            continue
        mapping[seq] = ch
    return mapping


def recompose_indic(text: str) -> str:
    """Restores precomposed Indic nukta letters after IndicNLP normalisation."""
    for seq, ch in _recomposition_map().items():
        if seq in text:
            text = text.replace(seq, ch)
    return text


def normalize_indic(text: str, language_iso_code: str) -> str:
    """Canonicalises Indic text before G2P.

    Indic scripts encode the same grapheme several ways, and espeak-ng only has rules for
    one of them. Verified on this machine:

        क़  U+0958 (precomposed)      -> U+0915 U+093C  (KA + NUKTA)
        ড়  U+09DC (precomposed)      -> U+09A1 U+09BC  (DDA + NUKTA)
        क्‍ष  with U+200D ZWJ            -> ZWJ removed

    Fed the precomposed or ZWJ-bearing form, espeak either skips the codepoint or emits
    something arbitrary — silently, and only for the subset of rows that happen to use that
    encoding. That is a per-row error concentrated in exactly the words most likely to be
    loanwords and proper nouns, which is worse than a uniform bias because it cannot be
    calibrated away.

    Normalising is a strict improvement and costs nothing: on already-canonical text it is
    a no-op (verified across all 11 languages). It runs inside `phonemize`, so labels and
    scores are normalised identically — the same-function rule (non-negotiable #3) extends
    to preprocessing, not just to the G2P call.
    """
    n = _normalizer(language_iso_code)
    if n is None:
        if "warned" not in _NORMALIZER_WARNED:
            _NORMALIZER_WARNED.add("warned")
            logger.warning(
                "indic-nlp-library is not installed — phoneme counts will be taken on "
                "UN-normalised text. Precomposed nukta forms and ZWJ sequences will be "
                "mis-phonemized for a minority of rows. Install with: "
                "pip install indic-nlp-library"
            )
        return text
    return recompose_indic(n.normalize(text))


# ---------------------------------------------------------------------------------------
# Backend access
# ---------------------------------------------------------------------------------------

@functools.lru_cache(maxsize=1)
def _backend_version() -> str:
    """Returns the espeak-ng version string, or raises G2PUnavailable.

    Cached because it shells into the espeak library, and callers ask for it once per
    artifact written.
    """
    try:
        from phonemizer.backend.espeak.wrapper import EspeakWrapper
    except ImportError as e:
        raise G2PUnavailable(f"`phonemizer` is not importable ({e}).\n{_INSTALL_HINT}") from e

    try:
        version = EspeakWrapper().version
    except Exception as e:  # noqa: BLE001 - any failure here means the shared library is unusable
        raise G2PUnavailable(
            f"`phonemizer` imported but the espeak-ng shared library is unusable ({e}).\n"
            f"{_INSTALL_HINT}"
        ) from e

    # Normalise: some phonemizer builds return a tuple like (1, 50) rather than "1.50".
    # `str()` on that yields "(1, 50)" — which embeds a COMMA and a space into a string
    # that gets stamped into every dataset row and every report header, and would corrupt
    # any CSV column it ever lands in. A provenance field that can break its own container
    # is not provenance.
    if isinstance(version, (tuple, list)):
        version = ".".join(str(p) for p in version)
    return re.sub(r"[\s,]+", "", str(version))


def ruler_id() -> str:
    """The identifier stamped into every dataset row and eval report this module touches.

    Raises G2PUnavailable rather than returning a placeholder — a report that cannot name
    its ruler must not be written at all.
    """
    return f"{RULER_PHONEMES}-{_backend_version()}"


# ---------------------------------------------------------------------------------------
# Core conversion
# ---------------------------------------------------------------------------------------

def _normalise_tokens(raw: str) -> list[str]:
    """Turns espeak's separated output into a list of countable phoneme symbols."""
    raw = _LANG_SWITCH_RE.sub(" ", raw)
    raw = raw.replace("|", " ").replace(_UNDERTIE, " ")
    tokens = []
    for tok in raw.split():
        tok = tok.strip(_STRESS_MARKS)
        # A token that was *only* stress marks collapses to empty and is not a sound.
        if tok:
            tokens.append(tok)
    return tokens


def phonemize(text: str, language_iso_code: str) -> list[str]:
    """Converts `text` to a list of IPA phoneme symbols. Never falls back.

    Raises:
        G2PUnavailable: espeak-ng / phonemizer is missing or broken.
        PhonemizationError: the backend works but produced nothing for this input.
    """
    text = (text or "").strip()
    if not text:
        return []

    _backend_version()  # raises G2PUnavailable with the install hint if the backend is dead

    from phonemizer import phonemize as _ph
    from phonemizer.separator import Separator

    lang = get_language(language_iso_code)
    text = normalize_indic(text, language_iso_code)
    try:
        out = _ph(
            text,
            language=lang.espeak_code,
            backend="espeak",
            separator=Separator(phone=" ", word=" | "),
            strip=True,
            njobs=1,
        )
    except Exception as e:  # noqa: BLE001
        raise PhonemizationError(
            f"espeak-ng failed on {language_iso_code} input {text[:60]!r}: {e}"
        ) from e

    tokens = _normalise_tokens(out)
    if not tokens:
        raise PhonemizationError(
            f"espeak-ng returned no phonemes for {language_iso_code} input {text[:60]!r}. "
            f"Raw backend output was {out[:80]!r}."
        )
    return tokens


def phonemize_many(texts: Sequence[str], language_iso_code: str) -> list[list[str]]:
    """Batched `phonemize`, one espeak call for the whole list.

    Roughly an order of magnitude faster than looping — which matters, because relabelling
    the 53,350-row corpus one string at a time is a multi-hour job and a batched pass is
    minutes. Empty inputs map to empty lists; a backend failure on the batch raises, so a
    partially-phonemized corpus is never written.
    """
    if not texts:
        return []

    _backend_version()

    from phonemizer import phonemize as _ph
    from phonemizer.separator import Separator

    lang = get_language(language_iso_code)
    cleaned = [normalize_indic((t or "").strip(), language_iso_code) for t in texts]
    try:
        out = _ph(
            cleaned,
            language=lang.espeak_code,
            backend="espeak",
            separator=Separator(phone=" ", word=" | "),
            strip=True,
            njobs=1,
        )
    except Exception as e:  # noqa: BLE001
        raise PhonemizationError(
            f"espeak-ng failed on a batch of {len(texts)} {language_iso_code} strings: {e}"
        ) from e

    if isinstance(out, str):  # phonemizer collapses a 1-element list to a bare string
        out = [out]
    return [_normalise_tokens(o) for o in out]


@functools.lru_cache(maxsize=200_000)
def count_phonemes(text: str, language_iso_code: str) -> int:
    """The one function that defines "how many phonemes is this". Labels and scores both
    call it, which is what makes their numbers comparable (non-negotiable #3)."""
    return len(phonemize(text, language_iso_code))


def phoneme_inventory(texts: Iterable[str], language_iso_code: str) -> "Counter":
    """The distribution of phoneme symbols a language's G2P actually produces.

    This is a *validation* instrument, not a pipeline metric — the pipeline only ever needs
    the per-sentence count. But a count cannot tell you whether the symbols being counted
    are phonemes at all, and an inventory can, in one glance.

    Worked example of what it catches: epitran's Marathi renders `कॅल्शियम` as `kəॅlɕijmə`,
    leaking U+0945 DEVANAGARI VOWEL SIGN CANDRA E — a *source script* character — straight
    into its own IPA output, because that codepoint has no entry in its map. The count still
    comes out looking reasonable. The inventory makes it obvious.
    """
    from collections import Counter
    inv = Counter()
    for toks in phonemize_many(list(texts), language_iso_code):
        inv.update(toks)
    return inv


def validate_inventory(inventory: "Counter", language_iso_code: str) -> list[str]:
    """Returns a list of problems found in a phoneme inventory; empty means clean.

    The check that matters: a symbol containing a character from the language's OWN script
    is not a phoneme — it is an unmapped source character that the G2P passed through
    untranslated. Its presence proves the converter has a hole, and tells you exactly which
    grapheme fell in.
    """
    import unicodedata

    token = _script_token(language_iso_code)
    problems = []
    total = sum(inventory.values()) or 1
    for sym, n in inventory.most_common():
        leaked = [c for c in sym if token in unicodedata.name(c, "")]
        if leaked:
            names = ", ".join(f"U+{ord(c):04X} {unicodedata.name(c, '?')}" for c in leaked)
            problems.append(
                f"{sym!r} ({n} occurrences, {100 * n / total:.2f}%) contains untranslated "
                f"source-script characters: {names}"
            )
    return problems


def count_chars(text: str) -> int:
    """Non-space character count — the *wrong* ruler, defined here explicitly so audit
    code can name it and detect it rather than reimplementing it three times."""
    return len([c for c in (text or "") if not c.isspace()])


# ---------------------------------------------------------------------------------------
# Preflight
# ---------------------------------------------------------------------------------------

# Short canaries in each language's own script. Any real sentence works; these are kept
# tiny so the preflight costs milliseconds.
_CANARIES: dict[str, str] = {
    "hi": "दूध में कैल्शियम है",
    "bn": "দুধে ক্যালসিয়াম আছে",
    "mr": "दुधात कॅल्शियम आहे",
    "gu": "દૂધમાં કેલ્શિયમ છે",
    "pa": "ਦੁੱਧ ਵਿੱਚ ਕੈਲਸ਼ੀਅਮ ਹੈ",
    "ta": "பாலில் கால்சியம் உள்ளது",
    "te": "పాలలో కాల్షియం ఉంది",
    "kn": "ಹಾಲಿನಲ್ಲಿ ಕ್ಯಾಲ್ಸಿಯಂ ಇದೆ",
    "ml": "പാലിൽ കാൽസ്യം ഉണ്ട്",
    "or": "ଦୁଧରେ କ୍ୟାଲସିୟମ ଅଛି",
    "as": "গাখীৰত কেলচিয়াম আছে",
}


def assert_g2p_available(languages: Optional[Iterable[str]] = None, verbose: bool = True) -> dict:
    """Preflight. Call this at the top of every notebook that writes labels or scores them.

    Checks three things, in increasing order of strictness:

    1. `phonemizer` imports and the espeak-ng shared library loads.
    2. Every requested language produces non-empty output.
    3. **The output is actually phonemes, not the input's own characters.** This is the
       check that matters. A backend that silently passes text through, or a caller that
       silently substitutes a character split, both produce plausible non-empty output —
       and that is precisely the failure that mislabelled this project's corpus. We assert
       that at least one returned symbol does not appear in the source string, which is
       guaranteed true for any Indic script rendered to IPA and false for passthrough.

    Returns a manifest dict suitable for writing next to any artifact produced afterwards.

    Raises:
        G2PUnavailable: with the install hint, if any check fails.
    """
    codes = list(languages) if languages is not None else list(LANGUAGES.keys())
    version = _backend_version()   # raises with install hint

    report: dict[str, dict] = {}
    failures: list[str] = []

    for code in codes:
        canary = _CANARIES.get(code)
        if canary is None:
            failures.append(f"{code}: no canary string defined in common/phonemes.py")
            continue
        try:
            tokens = phonemize(canary, code)
        except (G2PUnavailable, PhonemizationError) as e:
            failures.append(f"{code}: {e}")
            continue

        source_chars = set(canary)
        novel = [t for t in tokens if not set(t) <= source_chars]
        looks_like_passthrough = not novel

        # Partial leaks: individual unmapped graphemes riding through into the IPA. The
        # passthrough test above only catches total failure; this catches the holes.
        from collections import Counter
        leaks = validate_inventory(Counter(tokens), code)

        report[code] = {
            "voice": get_language(code).espeak_code,
            "n_phonemes": len(tokens),
            "n_chars": count_chars(canary),
            "sample": " ".join(tokens[:12]),
            "passthrough": looks_like_passthrough,
            "normalized": _normalizer(code) is not None,
            "leaks": leaks,
        }
        if looks_like_passthrough:
            failures.append(
                f"{code}: espeak returned symbols drawn entirely from the input's own "
                f"characters ({' '.join(tokens[:10])}) — this is a passthrough or a "
                f"character-split fallback, NOT phonemization."
            )
        for lk in leaks:
            failures.append(f"{code}: untranslated grapheme in G2P output — {lk}")

    if failures:
        raise G2PUnavailable(
            "G2P preflight FAILED — do not generate labels or scores in this session.\n"
            + "\n".join(f"  - {f}" for f in failures)
            + "\n\n" + _INSTALL_HINT
        )

    manifest = {
        "ruler": ruler_id(),
        "espeak_version": version,
        "languages": report,
    }
    if verbose:
        logger.info("G2P preflight PASSED — ruler=%s", manifest["ruler"])
        for code, r in report.items():
            logger.info(
                "  %-3s voice=%-3s %3d phonemes / %3d chars (ratio %.3f)  %s",
                code, r["voice"], r["n_phonemes"], r["n_chars"],
                r["n_phonemes"] / max(r["n_chars"], 1), r["sample"],
            )
    return manifest


if __name__ == "__main__":  # `python -m common.phonemes` as a standalone preflight
    import json
    import sys

    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    try:
        print(json.dumps(assert_g2p_available(), ensure_ascii=False, indent=2))
    except G2PUnavailable as e:
        print(str(e), file=sys.stderr)
        raise SystemExit(1)


In [ ]:
%%writefile /kaggle/working/pipeline_v3/translation/__init__.py


In [ ]:
%%writefile /kaggle/working/pipeline_v3/translation/semantic_gate.py
"""
translation/semantic_gate.py
-----------------------------------------
Inference-time meaning-preservation check for isochrony-constrained translation.

WHY THIS EXISTS
---------------
`isochrony_translation_v3.generate_and_select` produces several candidate translations at
different phoneme budgets ("direct" 1.0, "paraphrase" 0.85, "minimal" 0.65) and, when
none of them fit the segment's time window, refines by tightening the budget a further
0.75x per round for up to 3 rounds. A segment can therefore legitimately end up at
0.65 * 0.75^3 ~= 0.27 of the original phoneme budget.

Until this module existed, the winning candidate was chosen by `_score_candidate`, which
looks *only* at predicted duration versus target duration. Nothing anywhere in the
inference path asked whether the surviving text still meant the same thing.

The precise failure this creates is worth stating carefully, because the obvious version
of the criticism is wrong. `_score_candidate` does NOT simply prefer the shortest text —
it scores closeness to the target duration, so a wildly over-compressed candidate is
penalised for undershooting just as an over-long one is penalised for overshooting.

The real gap is subtler and worse: **two candidates of identical predicted duration score
identically, whether one preserved the meaning and the other dropped a clause.** Hitting
a phoneme budget by rephrasing more tersely and hitting it by deleting the subordinate
clause produce the same number of phonemes, therefore the same predicted milliseconds,
therefore the same score. Timing is simply blind to the distinction. When the budget is
tight enough that faithful compression cannot fit, the deleting variant is the one that
lands on target — and it wins on merit under the old scoring.

That is precisely the failure mode the *training* pipeline already defends against:
`training/length_augmentation.py` gates every generated paraphrase on cosine similarity
>= 0.80 against the human reference, explicitly "to stop the model teaching itself
'delete random words to hit a phoneme count'". This module carries the same guarantee
into inference, where there is no human reference to compare against.

CHOICE OF ANCHOR (the part that is easy to get wrong)
-----------------------------------------------------
At training time the anchor is the human reference translation. At inference no reference
exists, so we need a stand-in for "what this segment is supposed to mean". Two are
available and we use both, in order of preference:

  1. SAME-LANGUAGE ANCHOR (preferred): the "direct" candidate, generated at the full 1.0
     phoneme budget. It is the least-compressed translation we have and it is produced
     anyway, so it costs nothing extra. Comparing a compressed candidate against it
     measures exactly what we care about: "how much meaning did compression cost?"
     Same-language cosines are well calibrated and directly comparable to the training
     gate's 0.80 threshold.

  2. CROSS-LINGUAL FALLBACK: the English source. Used when the direct candidate failed to
     generate or was empty. The embedder is multilingual and aligns translations across
     languages, but cross-lingual cosines sit systematically lower than same-language
     ones for identical meaning, so this path uses its own (lower) threshold. Do not
     compare the two numbers to each other.

The embedder is deliberately the SAME model the training gate uses
(paraphrase-multilingual-MiniLM-L12-v2). If the two used different embedders, the
threshold learned/validated on one would not transfer to the other.

FAILURE POLICY
--------------
This gate never crashes a dub and never silently ships a bad line:
  - If `sentence-transformers` is not installed, or the model cannot load, the gate
    disables itself, logs ONE warning, and selection falls back to timing-only scoring
    (exactly the old behaviour). Semantic scores are reported as None, not as 1.0, so
    downstream logs can tell "not checked" apart from "checked and fine".
  - If every candidate scores below threshold, the pipeline does not fail. The candidate
    with the highest similarity among those that fit the window is chosen and the segment
    is flagged `semantic_degraded=True`. A dub with one weak line is recoverable; a
    crashed 40-minute render is not. The flag is what makes the weak line findable.
"""

from __future__ import annotations

import logging
import os
from typing import Optional, Sequence

logger = logging.getLogger(__name__)

# Must match training/length_augmentation.py — see module docstring.
DEFAULT_EMBEDDER_MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Same-language floor, aligned with the training gate's min_similarity=0.80.
DEFAULT_MIN_SIMILARITY_SAME_LANG = float(os.getenv("ISOCHRONY_MIN_SIM_SAME_LANG", "0.80"))

# Cross-lingual floor. Lower by construction: EN<->Indic cosines for identical meaning
# typically land ~0.60-0.75 with this embedder, well below same-language equivalents.
DEFAULT_MIN_SIMILARITY_CROSS_LANG = float(os.getenv("ISOCHRONY_MIN_SIM_CROSS_LANG", "0.60"))

# How much semantic fidelity counts relative to timing fit in the combined score.
DEFAULT_SEMANTIC_WEIGHT = float(os.getenv("ISOCHRONY_SEMANTIC_WEIGHT", "0.4"))

_DISABLED_WARNING_EMITTED = False


class SemanticGate:
    """Multilingual sentence embedder wrapper for meaning-preservation checks.

    Lazily loaded so importing this module costs nothing until a translation actually
    runs, and so a missing optional dependency degrades to "gate off" rather than an
    ImportError at module import time.
    """

    def __init__(self, model_id: str = DEFAULT_EMBEDDER_MODEL_ID,
                 device: Optional[str] = None, enabled: bool = True):
        self.model_id = model_id
        self.device = device
        self.enabled = enabled
        self._model = None
        self._load_failed = False

    def _lazy_load(self):
        global _DISABLED_WARNING_EMITTED
        if self._model is not None or self._load_failed or not self.enabled:
            return self._model
        try:
            from sentence_transformers import SentenceTransformer
            logger.info("Loading semantic fidelity embedder %s ...", self.model_id)
            self._model = SentenceTransformer(self.model_id, device=self.device)
        except Exception as e:  # noqa: BLE001
            self._load_failed = True
            if not _DISABLED_WARNING_EMITTED:
                _DISABLED_WARNING_EMITTED = True
                logger.warning(
                    "Semantic fidelity gate DISABLED (%s: %s). Candidate selection will "
                    "fall back to timing-only scoring, which cannot detect a translation "
                    "that hit its phoneme budget by dropping meaning. Install with: "
                    "pip install sentence-transformers",
                    type(e).__name__, e,
                )
        return self._model

    @property
    def available(self) -> bool:
        return self._lazy_load() is not None

    def similarities(self, anchor: str, candidates: Sequence[str]) -> Optional[list[float]]:
        """Cosine similarity of each candidate against the anchor.

        Returns None when the gate is unavailable, so callers can distinguish
        "not checked" from "checked and scored 0.0". One batched forward pass.
        """
        model = self._lazy_load()
        if model is None or not candidates:
            return None
        try:
            import numpy as np
            embeddings = model.encode(
                [anchor, *candidates], normalize_embeddings=True, show_progress_bar=False
            )
            anchor_vec = embeddings[0]
            return [float(np.dot(anchor_vec, v)) for v in embeddings[1:]]
        except Exception as e:  # noqa: BLE001
            logger.warning("Semantic similarity computation failed: %s", e, exc_info=True)
            return None

    def similarity(self, text_a: str, text_b: str) -> Optional[float]:
        result = self.similarities(text_a, [text_b])
        return result[0] if result else None


def combined_score(timing_score: float, semantic_similarity: Optional[float],
                   min_similarity: float,
                   semantic_weight: float = DEFAULT_SEMANTIC_WEIGHT) -> float:
    """Blend timing fit with meaning preservation.

    Timing-only scoring optimises for the shortest text that fits, which is the wrong
    objective — the goal is the *most faithful* text that fits, not the shortest.

    Candidates at or above `min_similarity` are ranked on a weighted blend. Candidates
    below it are pushed beneath every passing candidate but keep their relative order,
    so that if everything fails the floor we still select the least-bad option rather
    than an arbitrary one.

    When similarity is None (gate unavailable) this returns the timing score unchanged,
    preserving the previous behaviour exactly.
    """
    if semantic_similarity is None:
        return timing_score
    blended = (1.0 - semantic_weight) * timing_score + semantic_weight * semantic_similarity
    if semantic_similarity < min_similarity:
        return blended - 1.0
    return blended


In [ ]:
%%writefile /kaggle/working/pipeline_v3/evaluation/__init__.py


In [ ]:
%%writefile /kaggle/working/pipeline_v3/evaluation/phoneme_adherence_eval.py
"""
evaluation/phoneme_adherence_eval.py
=====================================
Checkpoint-trajectory evaluation for the length-constrained translation fine-tune.

WHAT CHANGED IN THIS REVISION, AND WHY
---------------------------------------
The previous harness produced numbers that could not be trusted, in four separate ways.
Each fix below corresponds to a defect that was found in its output, not to a style
preference.

**1. chrF++ was being swallowed.** `import sacrebleu` sat inside the same `try` as the
scoring call, under a bare `except Exception: return None`. When the pip install failed in
a Kaggle session, every one of 440 calls returned None without a single log line, the run
completed, a report was written, and the fidelity column came out blank — the one column
that would have told us whether the length constraint was being paid for out of meaning.
The import now happens once at module scope and its failure is logged loudly and recorded
in the report header, so an absent metric is visible rather than merely empty.

**2. There were two different slope estimators and the report used the wrong one.**
They are not duplicates; they measure different things, and the distinction is the whole
point of the metric:

  - *population slope* regresses generated length against requested length across
    **different sentences**, whose budgets differ because the sentences differ. A model
    that ignores the budget entirely still scores high on it — longer English produces
    longer Hindi regardless. It measures whether translations are appropriately scaled,
    which is a fluency property, not budget obedience.
  - *probe slope* holds the sentence **fixed** and sweeps only the requested budget across
    0.6-1.4x. The sentence is constant, so the only thing that can move the output length
    is the budget. This is the capability probe.

The old report quoted the population slope while the surrounding prose described the
probe. Both are now computed, named distinctly, and the **probe** is what the report
leads with.

**3. The probe's budgets were derived from the corpus labels**, which are known to be
mislabelled (see `common/phonemes.py`). Natural length is now measured from the reference
text with the canonical counter, so the probe is independent of the corpus labels and
measures capability against the true ruler.

**4. `summarize` used the population variance divisor and returned the upper-middle value
as the "median"** for even-length samples. Small, systematic, and in every median in every
report. Now sample variance (n-1) and a true median.

Two things were also missing rather than broken:

**Semantic fidelity was never measured.** chrF++ needs a reference translation, which
exists here and never exists at dub time. The production-side question — "how much meaning
did compression cost, measured against something available at inference?" — is answered by
scoring each generation against the full-budget candidate using the same embedder the
inference gate uses. That produces a degraded-segment rate per language, which is the
number that converts an architectural worry into evidence.

**The stopping rule lived in prose.** It is now `stopping_verdict()`, computed from the
trajectory and printed in the report, so the decision cannot be re-argued after the fact.

READING THE OUTPUT
------------------
Read the **per-language** table first, then the aggregate. Eleven languages across two
families do not plateau together — Dravidian languages are agglutinative, so their
token-to-phoneme relationship differs and they learn this task on a different schedule.
Every aggregate number hides that. Building the decomposition and then reading the
aggregate column anyway is the mistake this harness was already capable of preventing.
"""

from __future__ import annotations

import argparse
import glob
import json
import logging
import math
import os
import re
import statistics
import sys
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.languages import LANGUAGES, get_language  # noqa: E402
from common.phonemes import (  # noqa: E402
    PhonemizationError, assert_g2p_available, count_phonemes, ruler_id,
)

logger = logging.getLogger("phoneme_adherence_eval")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

PROMPT_RE = re.compile(r"\[Target Phonemes:\s*(\d+)\]")
LENGTH_SWEEP_FACTORS = [0.6, 0.8, 1.0, 1.2, 1.4]

# --- chrF++ availability, resolved once, loudly -----------------------------------------
# Imported at module scope precisely so its absence is a visible fact about the run rather
# than 440 silent Nones.
try:
    import sacrebleu as _sacrebleu
    CHRF_AVAILABLE = True
    CHRF_UNAVAILABLE_REASON = None
except Exception as _e:  # noqa: BLE001
    _sacrebleu = None
    CHRF_AVAILABLE = False
    CHRF_UNAVAILABLE_REASON = repr(_e)
    logger.error(
        "sacrebleu is NOT importable (%s). chrF++ will be absent from this report. "
        "Fidelity is the axis a length-targeted fine-tune puts at risk, so a run without "
        "it answers a strictly smaller question. Install with: pip install sacrebleu",
        CHRF_UNAVAILABLE_REASON,
    )


# ========================================================================================
# Corpus IO
# ========================================================================================

def read_val(path: str) -> list:
    rows, bad = [], 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                bad += 1
    if bad:
        logger.warning("%d unparseable lines skipped in %s", bad, path)
    return rows


def group_by_language(rows: list) -> dict:
    g = defaultdict(list)
    for r in rows:
        g[r.get("language", "unknown")].append(r)
    return g


def requested_n(row: dict) -> int:
    """The budget the row's own prompt states — i.e. what the model was actually asked for.

    Read from the prompt text first, not from the `n_phonemes` field, because the prompt
    is what the model sees. If a relabelling ever updates one and not the other, this
    reports the number that actually conditioned the generation.
    """
    m = PROMPT_RE.search(row.get("prompt", ""))
    if m:
        return int(m.group(1))
    return int(row.get("n_phonemes") or 0)


def true_phoneme_len(text: str, lang: str) -> Optional[int]:
    try:
        return count_phonemes(text, lang)
    except PhonemizationError as e:
        logger.warning("phonemization failed (%s): %s", lang, e)
        return None


# ========================================================================================
# Statistics
# ========================================================================================

def summarize(xs: list) -> dict:
    """Sample statistics. Sample stdev (n-1) and a true median — the previous version used
    the population divisor and `xs_sorted[n // 2]`, which returns the upper middle value
    for even n."""
    if not xs:
        return {"n": 0}
    n = len(xs)
    return {
        "n": n,
        "mean": statistics.fmean(xs),
        "median": statistics.median(xs),
        "std": statistics.stdev(xs) if n > 1 else 0.0,
    }


def linfit_slope(xs: list, ys: list):
    n = len(xs)
    if n < 2:
        return None, None
    mx, my = sum(xs) / n, sum(ys) / n
    sxx = sum((x - mx) ** 2 for x in xs)
    sxy = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    if sxx == 0:
        return None, None
    slope = sxy / sxx
    syy = sum((y - my) ** 2 for y in ys)
    r2 = (sxy * sxy) / (sxx * syy) if syy > 0 else None
    return slope, r2


def chrf_pp(hypothesis: str, reference: str) -> Optional[float]:
    """chrF++ against the reference. Returns None only when sacrebleu is genuinely absent
    — which is recorded once, at module import, rather than per call."""
    if not CHRF_AVAILABLE:
        return None
    return float(_sacrebleu.sentence_chrf(hypothesis, [reference], word_order=2).score)


# ========================================================================================
# Semantic scoring — the production-side fidelity check
# ========================================================================================

class SemanticScorer:
    """Cosine similarity with the same embedder the inference gate uses.

    Deliberately the same model as `translation/semantic_gate.py` and
    `training/length_augmentation.py`: a threshold validated on one embedder means nothing
    on another, so an eval that used a different one could not be compared against the
    gate that ships.
    """

    def __init__(self, model_id: Optional[str] = None, device: Optional[str] = None):
        from translation.semantic_gate import DEFAULT_EMBEDDER_MODEL_ID
        self.model_id = model_id or DEFAULT_EMBEDDER_MODEL_ID
        self.device = device
        self._model = None
        self.available = True
        self.reason = None

    def _lazy(self):
        if self._model is None and self.available:
            try:
                from sentence_transformers import SentenceTransformer
                logger.info("Loading semantic embedder %s ...", self.model_id)
                self._model = SentenceTransformer(self.model_id, device=self.device)
            except Exception as e:  # noqa: BLE001
                self.available = False
                self.reason = repr(e)
                logger.error("Semantic scoring DISABLED — embedder failed to load: %s", e)
        return self._model

    def similarity(self, a: str, b: str) -> Optional[float]:
        model = self._lazy()
        if model is None or not a or not b:
            return None
        import numpy as np
        emb = model.encode([a, b], normalize_embeddings=True)
        return float(np.dot(emb[0], emb[1]))


# ========================================================================================
# Model loading and generation
# ========================================================================================

def load_model(adapter_path: Optional[str], base_model_id: str, max_seq_length: int = 512):
    from unsloth import FastLanguageModel
    src = adapter_path if adapter_path else base_model_id
    model, tok = FastLanguageModel.from_pretrained(
        src, max_seq_length=max_seq_length, load_in_4bit=True, dtype=None,
    )
    FastLanguageModel.for_inference(model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return model, tok


def _clean(text: str) -> str:
    text = text.strip()
    for prefix in ("Translation:", "Output:", "Target:"):
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    if len(text) >= 2 and text[0] in "\"'" and text[-1] in "\"'":
        text = text[1:-1].strip()
    return text


def generate_batch(model, tok, prompts: list[str], max_new_tokens: int = 128,
                   temperature: float = 0.3, batch_size: int = 8) -> list[str]:
    """Batched generation.

    The probe needs 5 generations per sentence per language per checkpoint; unbatched that
    dominates the entire session's wall clock and is the reason the probe was previously
    run at 10 sentences per language, where per-language orderings are not trustworthy.
    Left padding is required — decoder-only models continue from the rightmost token, and
    right padding would have the model continue from pad tokens.
    """
    import torch
    outs: list[str] = []
    prev_side = tok.padding_side
    tok.padding_side = "left"
    try:
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            texts = [
                tok.apply_chat_template([{"role": "user", "content": p}],
                                        tokenize=False, add_generation_prompt=True)
                for p in chunk
            ]
            enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                gen = model.generate(
                    **enc, max_new_tokens=max_new_tokens,
                    do_sample=temperature > 0, temperature=max(temperature, 1e-4),
                    top_p=0.9, pad_token_id=tok.pad_token_id,
                )
            for j in range(len(chunk)):
                new = gen[j][enc["input_ids"].shape[1]:]
                outs.append(_clean(tok.decode(new, skip_special_tokens=True)))
    finally:
        tok.padding_side = prev_side
    return outs


def completion_ce(model, tok, prompt: str, completion: str, max_seq_length: int = 512) -> Optional[float]:
    import torch
    ids = tok.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": completion}],
        return_tensors="pt", add_generation_prompt=False,
    )
    prompt_ids = tok.apply_chat_template(
        [{"role": "user", "content": prompt}], return_tensors="pt", add_generation_prompt=True,
    )
    if ids.shape[1] > max_seq_length:
        return None
    labels = ids.clone()
    labels[:, : prompt_ids.shape[1]] = -100
    ids, labels = ids.to(model.device), labels.to(model.device)
    with torch.no_grad():
        loss = model(input_ids=ids, labels=labels).loss
    return float(loss.item())


# ========================================================================================
# Metric passes
# ========================================================================================

def eval_checkpoint(model, tok, rows_by_lang: dict, adherence_per_lang: int,
                    ce_per_lang: int, do_generation: bool, do_ce: bool,
                    semantic: Optional[SemanticScorer] = None,
                    semantic_threshold: float = 0.80,
                    budget_scale: Optional[dict] = None,
                    batch_size: int = 8,
                    dump_langs: Optional[set] = None, dump_n: int = 0,
                    dump_sink: Optional[list] = None, ckpt_label: str = "") -> dict:
    results = {}
    dump_langs = dump_langs or set()

    for lang, rows in rows_by_lang.items():
        if lang not in LANGUAGES:
            continue
        lang_res = {"language": lang}

        if do_ce:
            ce_vals = []
            for r in rows[:ce_per_lang]:
                c = completion_ce(model, tok, r["prompt"], r.get("completion") or r["target"])
                if c is not None:
                    ce_vals.append(c)
            s = summarize(ce_vals)
            lang_res["ce_mean"] = s.get("mean")
            lang_res["ce_perplexity"] = math.exp(s["mean"]) if s.get("mean") is not None else None
            lang_res["ce_n"] = s["n"]

        if do_generation:
            subset = [r for r in rows[:adherence_per_lang] if requested_n(r) > 0]
            prompts = [_rescaled_prompt(r, lang, budget_scale) for r in subset]
            gens = generate_batch(model, tok, prompts, temperature=0.3, batch_size=batch_size)

            rel_errs, signed_errs, chrfs, sims = [], [], [], []
            req_ns, gen_ns = [], []
            degraded = 0
            dumped = 0
            for r, gen in zip(subset, gens):
                if not gen:
                    continue
                # The budget we hold the model to is always in TRUE phonemes, whatever unit
                # the corpus label happened to be written in.
                N = _true_budget(r, lang)
                if not N:
                    continue
                n_gen = true_phoneme_len(gen, lang)
                if n_gen is None:
                    continue
                ref = r.get("target") or r.get("completion") or ""
                rel_errs.append(abs(n_gen - N) / N)
                signed_errs.append((n_gen - N) / N)
                req_ns.append(float(N))
                gen_ns.append(float(n_gen))

                c = chrf_pp(gen, ref) if ref else None
                if c is not None:
                    chrfs.append(c)

                sim = None
                if semantic is not None:
                    # Anchor = the human reference. At dub time no reference exists and the
                    # anchor is the full-budget candidate instead (see probe below); here the
                    # reference is the stronger anchor and is free.
                    sim = semantic.similarity(ref, gen) if ref else None
                    if sim is not None:
                        sims.append(sim)
                        if sim < semantic_threshold:
                            degraded += 1

                if lang in dump_langs and dump_sink is not None and dumped < dump_n:
                    dump_sink.append({
                        "checkpoint": ckpt_label, "language": lang, "english": r.get("english"),
                        "requested_n": N, "generated_n": n_gen, "generated": gen,
                        "reference": ref, "chrf": c, "semantic_similarity": sim,
                    })
                    dumped += 1

            ra, sa, ca, si = summarize(rel_errs), summarize(signed_errs), summarize(chrfs), summarize(sims)
            lang_res["adherence_rel_mean"] = ra.get("mean")
            lang_res["adherence_rel_median"] = ra.get("median")
            lang_res["adherence_signed_mean"] = sa.get("mean")
            lang_res["adherence_signed_median"] = sa.get("median")
            lang_res["chrf_mean"] = ca.get("mean")
            lang_res["semantic_mean"] = si.get("mean")
            lang_res["semantic_degraded_frac"] = (degraded / si["n"]) if si.get("n") else None
            lang_res["adherence_n"] = ra["n"]
            slope, r2 = linfit_slope(req_ns, gen_ns)
            # Named for what it is. Across DIFFERENT sentences, so it is confounded by
            # sentence length and is a fluency proxy, not a budget-obedience measurement.
            lang_res["length_slope_population"] = slope
            lang_res["length_r2_population"] = r2

        results[lang] = lang_res
    return results


def _true_budget(row: dict, lang: str) -> Optional[int]:
    """The budget in true phonemes.

    If the row was written by the repaired labeller it carries `ruler`, and `n_phonemes`
    is already correct. Otherwise the reference text is re-counted, so a legacy
    character-ruled corpus is still evaluated on the right ruler.
    """
    if str(row.get("ruler", "")).startswith("phonemes:"):
        n = int(row.get("n_phonemes") or 0)
        if n > 0:
            return n
    ref = row.get("target") or row.get("completion") or ""
    return true_phoneme_len(ref, lang) if ref else None


def _rescaled_prompt(row: dict, lang: str, budget_scale: Optional[dict]) -> str:
    """The prompt to send.

    With `--budget_scale_json`, the true-phoneme budget is divided by that language's
    phonemes-per-character constant before being written into the prompt. That converts
    "the budget I want, in phonemes" into "the budget this model was actually taught, in
    characters" — the salvage path for a checkpoint trained on character-ruled labels,
    which needs no retraining. Without the flag the row's own prompt is used unchanged.
    """
    if not budget_scale or lang not in budget_scale:
        return row["prompt"]
    N = _true_budget(row, lang)
    if not N:
        return row["prompt"]
    k = budget_scale[lang]
    asked = max(1, round(N / k)) if k else N
    return PROMPT_RE.sub(f"[Target Phonemes: {asked}]", row["prompt"])


def length_response_probe(model, tok, rows_by_lang: dict, sentences_per_lang: int,
                          semantic: Optional[SemanticScorer] = None,
                          semantic_threshold: float = 0.80,
                          budget_scale: Optional[dict] = None,
                          batch_size: int = 8,
                          points_sink: Optional[list] = None) -> dict:
    """The capability probe: one sentence, five budgets, only the budget varies.

    Natural length is measured from the reference text with the canonical counter rather
    than read from the corpus label, so the probe is unaffected by how the corpus was
    labelled.

    The semantic anchor here is the model's own 1.0x generation — the same anchor the
    inference gate uses, because at dub time no reference exists. That makes the degraded
    rate reported here directly comparable to what the shipped gate will see.

    `points_sink` collects the raw (requested, produced) pairs. Session 02i stored only the
    fitted slope and R², and that turned out to be too little: five languages came back at
    slope ~0.4 with R² ~0.55, and a summary statistic cannot distinguish "flat across the
    whole sweep" from "follows the budget down to 0.8x and then saturates". Those are
    different defects with different fixes.
    """
    out = {}
    for lang, rows in rows_by_lang.items():
        if lang not in LANGUAGES:
            continue
        lang_name = get_language(lang).name

        specs = []          # (sentence_index, factor, prompt)
        naturals = []
        for si, r in enumerate(rows[:sentences_per_lang]):
            eng = r.get("english") or ""
            ref = r.get("target") or r.get("completion") or ""
            if not eng or not ref:
                continue
            natural = true_phoneme_len(ref, lang)
            if not natural:
                continue
            naturals.append(natural)
            k = (budget_scale or {}).get(lang)
            for f in LENGTH_SWEEP_FACTORS:
                want = max(1, round(natural * f))          # what we want, in phonemes
                asked = max(1, round(want / k)) if k else want   # what we write in the prompt
                specs.append((len(naturals) - 1, f, want,
                              f'[Translate to {lang_name}] [Target Phonemes: {asked}] "{eng}"'))

        if not specs:
            out[lang] = {"language": lang, "length_slope_probe": None, "n_points": 0}
            continue

        gens = generate_batch(model, tok, [s[3] for s in specs], temperature=0.3,
                              batch_size=batch_size)

        req_all, gen_all = [], []
        norm_x, norm_y = [], []       # scale vs produced/natural — see the note below
        by_sentence: dict[int, dict[float, str]] = defaultdict(dict)
        for (si, f, want, _), gen in zip(specs, gens):
            if not gen:
                continue
            n_gen = true_phoneme_len(gen, lang)
            if n_gen is None:
                continue
            req_all.append(float(want))
            gen_all.append(float(n_gen))
            if naturals[si] > 0:
                norm_x.append(float(f))
                norm_y.append(n_gen / naturals[si])
            by_sentence[si][f] = gen
            if points_sink is not None:
                points_sink.append({
                    "language": lang, "sentence_idx": si, "scale": f,
                    "natural_n": naturals[si], "requested_n": want,
                    "produced_n": n_gen,
                })

        slope, r2 = linfit_slope(req_all, gen_all)

        # `slope` above is NOT zero-referenced, and that is easy to miss. It regresses raw
        # produced length on raw requested length, pooling points from sentences of
        # different lengths — and since requested = scale x natural, the spread of natural
        # leaks into the fit. A model that emits its natural translation every time,
        # ignoring the budget completely, therefore scores 0.60-0.83 rather than 0, with a
        # different value in every language. Reading it against 0 overstates every model.
        #
        # So report the floor next to it, and report the within-sentence estimator, in
        # which natural divides out and 0 really does mean "ignored the budget".
        floor, _ = linfit_slope(
            [float(max(1, round(naturals[si] * f))) for si in range(len(naturals))
             for f in LENGTH_SWEEP_FACTORS],
            [float(naturals[si]) for si in range(len(naturals))
             for _ in LENGTH_SWEEP_FACTORS])
        slope_norm, r2_norm = linfit_slope(norm_x, norm_y)

        # Semantic cost of compression, anchored the way production anchors it.
        sims, degraded, n_scored = [], 0, 0
        if semantic is not None:
            for si, byf in by_sentence.items():
                anchor = byf.get(1.0)
                if not anchor:
                    continue
                for f, gen in byf.items():
                    if f >= 1.0:
                        continue
                    s = semantic.similarity(anchor, gen)
                    if s is None:
                        continue
                    sims.append(s)
                    n_scored += 1
                    if s < semantic_threshold:
                        degraded += 1

        out[lang] = {
            "language": lang,
            # The one to read. Zero-referenced: 0 = ignores the budget, 1 = follows it.
            "length_slope_normalized": slope_norm,
            "length_r2_normalized": r2_norm,
            # Kept for continuity with earlier reports, with the floor beside it so the
            # number cannot be read against zero by mistake.
            "length_slope_probe": slope,
            "length_r2_probe": r2,
            "length_slope_probe_floor": floor,
            "length_slope_probe_above_floor": (
                (slope - floor) / (1.0 - floor)
                if slope is not None and floor is not None and floor < 1.0 else None),
            "n_points": len(req_all),
            "n_sentences": len(naturals),
            "compressed_semantic_mean": statistics.fmean(sims) if sims else None,
            "compressed_degraded_frac": (degraded / n_scored) if n_scored else None,
        }
    return out


# ========================================================================================
# The stopping rule, as code
# ========================================================================================

def stopping_verdict(traj: list[dict], ce_flat_tol: float = 0.005,
                     slope_move_tol: float = 0.01) -> dict:
    """CE-flat + slope still moving => keep spending quota.
       CE-flat + slope flat        => genuine plateau; early-stop loses nothing.

    Written as code rather than kept in prose because the whole point of the rule is that
    it must survive the moment when stopping looks attractive. Uses the probe slope; the
    population slope is not a capability measurement.
    """
    pts = sorted([t for t in traj if t.get("step", -1) >= 0], key=lambda x: x["step"])
    if len(pts) < 2:
        return {"verdict": "INSUFFICIENT_DATA", "reason": "need at least two checkpoints"}

    def last_valid(key):
        vals = [(t["step"], t[key]) for t in pts if t.get(key) is not None]
        return vals[-2:] if len(vals) >= 2 else None

    ce = last_valid("ce_mean")
    sl = last_valid("length_slope_probe") or last_valid("length_slope_population")
    if sl is None:
        return {"verdict": "INSUFFICIENT_DATA", "reason": "no slope measured on >=2 checkpoints"}

    ce_delta = (ce[1][1] - ce[0][1]) if ce else None
    slope_delta = sl[1][1] - sl[0][1]
    ce_flat = ce_delta is None or abs(ce_delta) < ce_flat_tol
    slope_moving = slope_delta > slope_move_tol

    if ce_flat and slope_moving:
        verdict, reason = "CONTINUE", (
            f"CE flat (delta {ce_delta:+.4f}) but probe slope still climbing "
            f"({sl[0][1]:.3f} -> {sl[1][1]:.3f}, delta {slope_delta:+.3f}). Length control "
            f"is still being learned after the loss curve went quiet. Keep spending quota."
        )
    elif ce_flat:
        verdict, reason = "STOP", (
            f"CE flat (delta {ce_delta if ce_delta is None else f'{ce_delta:+.4f}'}) and probe "
            f"slope flat ({sl[0][1]:.3f} -> {sl[1][1]:.3f}, delta {slope_delta:+.3f}). "
            f"Genuine plateau; early-stopping loses nothing measurable."
        )
    else:
        verdict, reason = "CONTINUE", (
            f"CE still moving (delta {ce_delta:+.4f}). Not a plateau."
        )
    return {"verdict": verdict, "reason": reason,
            "ce_delta": ce_delta, "slope_delta": slope_delta,
            "steps_compared": [sl[0][0], sl[1][0]]}


# ========================================================================================
# Driver
# ========================================================================================

def checkpoint_step(path: str) -> int:
    m = re.search(r"checkpoint-(\d+)", os.path.basename(str(path).rstrip("/")))
    return int(m.group(1)) if m else -1


def run(args):
    # An eval that silently used a character fallback would report a mislabelled corpus as
    # correctly labelled. Refuse to start.
    g2p = assert_g2p_available()
    logger.info("G2P ruler: %s", g2p["ruler"])

    budget_scale = None
    if args.budget_scale_json:
        budget_scale = json.loads(Path(args.budget_scale_json).read_text(encoding="utf-8"))
        if "per_language" in budget_scale:  # accept a ruler_audit report directly
            budget_scale = {k: v["ols_k_through_origin"]
                            for k, v in budget_scale["per_language"].items()}
        logger.info("Budget rescale ACTIVE: %s", budget_scale)

    rows = read_val(args.val_jsonl)
    rows_by_lang = group_by_language(rows)
    logger.info("Loaded %d val rows across %d languages", len(rows), len(rows_by_lang))

    semantic = None
    if not args.no_semantic:
        semantic = SemanticScorer(device=args.semantic_device)

    targets = []
    if args.base_baseline:
        targets.append(("base_model", None))
    ckpts = list(args.checkpoints or [])
    if args.checkpoints_glob:
        found = glob.glob(args.checkpoints_glob)
        if not found:
            # The silent-resume bug's twin: a glob that matches nothing produces a run that
            # looks healthy and evaluates nothing.
            raise SystemExit(
                f"--checkpoints_glob {args.checkpoints_glob!r} matched NOTHING. "
                f"Check the directory depth before spending a session on it."
            )
        ckpts += found
    ckpts = sorted(set(ckpts), key=checkpoint_step)
    for c in ckpts:
        targets.append((os.path.basename(str(c).rstrip("/")), c))
    if not targets:
        raise SystemExit("Nothing to evaluate: pass --checkpoints/--checkpoints_glob and/or --base_baseline")

    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    do_gen = args.mode in ("all", "adherence", "length")
    do_ce = args.mode in ("all", "ce")
    do_probe = args.mode in ("all", "length")

    per_ckpt_rows, traj_rows, lr_rows, sample_rows = [], [], [], []
    probe_point_rows: list[dict] = []
    dump_langs = set(args.dump_samples_langs or [])

    wb = wandb_init({"ruler": g2p["ruler"], "val_jsonl": args.val_jsonl, "mode": args.mode,
                     "checkpoints": [t[0] for t in targets],
                     "budget_scale": budget_scale}, args.wandb_project,
                    args.wandb_entity, args.wandb_run_name,
                    required=args.wandb_required) if args.wandb else None

    for label, adapter in targets:
        logger.info("=== Evaluating %s ===", label)
        model, tok = load_model(adapter, args.base_model_id, args.max_seq_length)
        res = eval_checkpoint(
            model, tok, rows_by_lang, args.adherence_samples_per_lang,
            args.ce_samples_per_lang, do_gen, do_ce,
            semantic=semantic, semantic_threshold=args.semantic_threshold,
            budget_scale=budget_scale, batch_size=args.batch_size,
            dump_langs=dump_langs, dump_n=args.dump_samples_n,
            dump_sink=sample_rows, ckpt_label=label,
        )
        ckpt_points: list[dict] = []
        probe = length_response_probe(
            model, tok, rows_by_lang, args.probe_sentences_per_lang,
            semantic=semantic, semantic_threshold=args.semantic_threshold,
            budget_scale=budget_scale, batch_size=args.batch_size,
            points_sink=ckpt_points,
        ) if do_probe else {}
        probe_point_rows += [{"checkpoint": label, "step": checkpoint_step(adapter)
                              if adapter else -1, **p} for p in ckpt_points]

        def agg(source: dict, key: str):
            vals = [v[key] for v in source.values() if v.get(key) is not None]
            return sum(vals) / len(vals) if vals else None

        step = checkpoint_step(adapter) if adapter else -1
        traj_rows.append({
            "checkpoint": label, "step": step,
            "ce_mean": agg(res, "ce_mean"), "ce_perplexity": agg(res, "ce_perplexity"),
            "adherence_rel_mean": agg(res, "adherence_rel_mean"),
            "adherence_signed_mean": agg(res, "adherence_signed_mean"),
            "chrf_mean": agg(res, "chrf_mean"),
            "semantic_mean": agg(res, "semantic_mean"),
            "semantic_degraded_frac": agg(res, "semantic_degraded_frac"),
            "length_slope_population": agg(res, "length_slope_population"),
            "length_slope_probe": agg(probe, "length_slope_probe") if probe else None,
            "compressed_degraded_frac": agg(probe, "compressed_degraded_frac") if probe else None,
        })
        for lang, v in res.items():
            merged = {**v, **{k: val for k, val in (probe.get(lang) or {}).items()
                              if k != "language"}}
            per_ckpt_rows.append({"checkpoint": label, "step": step, **merged})
        for lang, v in probe.items():
            lr_rows.append({"checkpoint": label, "step": step, **v})

        _write_csv(out_dir / "per_checkpoint_metrics.csv", per_ckpt_rows)
        _write_csv(out_dir / "trajectory_summary.csv", traj_rows)
        if lr_rows:
            _write_csv(out_dir / "length_response.csv", lr_rows)
        if probe_point_rows:
            _write_csv(out_dir / "length_response_points.csv", probe_point_rows)
        # Stream to W&B on the same cadence as the CSVs — this is the only signal
        # visible while the session is still running.
        wandb_log_checkpoint(wb, traj_rows[-1],
                             [r for r in per_ckpt_rows if r["step"] == step])

        del model
        try:
            import gc
            import torch
            gc.collect(); torch.cuda.empty_cache()
        except Exception:  # noqa: BLE001
            pass

    if sample_rows:
        with open(out_dir / "samples.jsonl", "w", encoding="utf-8") as f:
            for row in sample_rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    manifest = {
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "ruler": g2p["ruler"],
        "val_jsonl": args.val_jsonl,
        "n_val_rows": len(rows),
        "mode": args.mode,
        "adherence_samples_per_lang": args.adherence_samples_per_lang,
        "probe_sentences_per_lang": args.probe_sentences_per_lang,
        "chrf_available": CHRF_AVAILABLE,
        "chrf_unavailable_reason": CHRF_UNAVAILABLE_REASON,
        "semantic_available": bool(semantic and semantic.available),
        "semantic_threshold": args.semantic_threshold,
        "budget_scale": budget_scale,
        "checkpoints": [t[0] for t in targets],
    }
    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    _write_report(out_dir / "eval_report.md", traj_rows, per_ckpt_rows, lr_rows, manifest)
    if wb is not None:
        try:
            import wandb as _wb
            cols = ["checkpoint", "step", "language", "adherence_rel_mean",
                    "adherence_signed_mean", "length_slope_probe",
                    "length_slope_population", "chrf_mean", "semantic_mean",
                    "semantic_degraded_frac", "ce_mean"]
            tbl = _wb.Table(columns=cols)
            for r in per_ckpt_rows:
                tbl.add_data(*[r.get(c) for c in cols])
            wb.log({"per_language": tbl})
            v = stopping_verdict(traj_rows)
            wb.summary["stopping_verdict"] = v["verdict"]
            wb.summary["stopping_reason"] = v.get("reason", "")
            wb.summary["ruler"] = manifest.get("ruler")
            wb.finish()
            logger.info("wandb run CLOSED")
        except Exception as e:  # noqa: BLE001
            logger.error("wandb finalisation failed: %s", e)
    logger.info("Wrote outputs to %s", out_dir)


def wandb_init(manifest: dict, project: str, entity: Optional[str],
               run_name: Optional[str], required: bool = False):
    """Opens the W&B run BEFORE evaluation starts, so metrics can stream.

    The first version of this logged everything in one call at the end of `run()`. That
    makes W&B useless for its actual job here: Kaggle publishes a notebook's log only when
    the session ends, so W&B is the only live signal during a multi-hour run — and a
    channel that reports nothing until the run is over is not a live signal. Metrics are
    now logged after each checkpoint completes, matching the CSV writes.

    `required=True` turns every failure below into a hard exit. Session 02i is why: Kaggle's
    secrets service returned a connection error, the notebook logged a warning and carried
    on, and 12 GPU-hours ran with no observability at all. "The artifacts land on disk
    anyway" is a fair argument for a five-minute local run and a bad one for a session that
    costs a third of the weekly quota. Pass it for anything running on Kaggle.
    """
    def _fail(msg: str):
        if required:
            raise SystemExit(
                f"W&B is required for this run and could not start: {msg}\n"
                f"  This is fatal by design — a multi-hour GPU session with no live channel\n"
                f"  is unobservable until it ends. Fix the credential and re-push, or drop\n"
                f"  --wandb_required if you accept running blind."
            )
        logger.error("%s — continuing without it. The evaluation artifacts on disk are "
                     "unaffected.", msg)
        return None

    try:
        import wandb
    except ImportError:
        return _fail("wandb not installed (pip install wandb)")

    # Check the credential before anything expensive. Without this the run discovers the
    # problem after loading an 8B model, or not at all.
    # Both filenames: wandb writes `_netrc` on Windows and `.netrc` everywhere else, and
    # checking only the POSIX name makes this abort on the one machine it runs from.
    if required and not (os.environ.get("WANDB_API_KEY") or
                         any((Path.home() / n).exists() for n in (".netrc", "_netrc"))):
        return _fail("no WANDB_API_KEY in the environment and no ~/.netrc or ~/_netrc")

    try:
        run = wandb.init(project=project, entity=entity, name=run_name,
                         job_type="evaluation", config=manifest, reinit=True)
        logger.info("wandb run OPEN: %s", getattr(run, "url", ""))
        return run
    except Exception as e:  # noqa: BLE001
        return _fail(f"wandb.init failed ({e})")


def wandb_log_checkpoint(run, traj_row: dict, lang_rows: list[dict]) -> None:
    """Streams one checkpoint's metrics as soon as it finishes."""
    if run is None:
        return
    step = traj_row["step"] if traj_row["step"] >= 0 else 0
    payload = {f"agg/{k}": v for k, v in traj_row.items()
               if k not in ("checkpoint", "step") and v is not None}
    for r in lang_rows:
        lang = r.get("language")
        for k in ("adherence_rel_mean", "adherence_signed_mean", "length_slope_probe",
                  "length_slope_population", "chrf_mean", "semantic_mean",
                  "semantic_degraded_frac", "ce_mean"):
            if r.get(k) is not None:
                payload[f"{lang}/{k}"] = r[k]
    try:
        run.log(payload, step=step)
        logger.info("wandb: logged %d metrics at step %d", len(payload), step)
    except Exception as e:  # noqa: BLE001
        logger.error("wandb log failed at step %s: %s", step, e)


def log_to_wandb(traj: list[dict], per_ckpt: list[dict], manifest: dict,
                 project: str, entity: Optional[str], run_name: Optional[str]) -> None:
    """Mirrors the report into Weights & Biases.

    What gets logged is deliberately not what was logged last time. CE and perplexity go up
    as *diagnostics*; the panels that matter are `length_slope_probe`, signed adherence, and
    the semantic degraded rate — the metrics that measure the objective. Logging CE
    prominently is how a run gets stopped on the wrong signal, and this project has already
    paid for that once.

    Per-language series are logged individually (`as/length_slope_probe`, …) because the
    aggregate hides that eleven languages across two families do not plateau together.
    """
    try:
        import wandb
    except ImportError:
        logger.warning("wandb not installed — skipping (pip install wandb)")
        return

    try:
        run = wandb.init(project=project, entity=entity, name=run_name,
                         job_type="evaluation", config=manifest, reinit=True)
    except Exception as e:  # noqa: BLE001
        logger.error("wandb.init failed (%s) — continuing without it. The evaluation "
                     "artifacts on disk are unaffected.", e)
        return

    for t in sorted(traj, key=lambda x: x["step"]):
        step = t["step"] if t["step"] >= 0 else 0
        payload = {f"agg/{k}": v for k, v in t.items()
                   if k not in ("checkpoint", "step") and v is not None}
        for r in per_ckpt:
            if r["step"] != t["step"]:
                continue
            lang = r.get("language")
            for k in ("adherence_rel_mean", "adherence_signed_mean", "length_slope_probe",
                      "length_slope_population", "chrf_mean", "semantic_mean",
                      "semantic_degraded_frac", "ce_mean"):
                if r.get(k) is not None:
                    payload[f"{lang}/{k}"] = r[k]
        run.log(payload, step=step)

    cols = ["checkpoint", "step", "language", "adherence_rel_mean", "adherence_signed_mean",
            "length_slope_probe", "length_slope_population", "chrf_mean", "semantic_mean",
            "semantic_degraded_frac", "ce_mean"]
    tbl = wandb.Table(columns=cols)
    for r in per_ckpt:
        tbl.add_data(*[r.get(c) for c in cols])
    run.log({"per_language": tbl})

    v = stopping_verdict(traj)
    run.summary["stopping_verdict"] = v["verdict"]
    run.summary["stopping_reason"] = v.get("reason", "")
    run.summary["ruler"] = manifest.get("ruler")
    run.finish()
    logger.info("wandb run: %s", getattr(run, "url", ""))


def _write_csv(path, rows):
    if not rows:
        return
    cols = list({k for r in rows for k in r.keys()})
    order = ["checkpoint", "step", "language"]
    cols = [c for c in order if c in cols] + sorted(c for c in cols if c not in order)
    # csv.writer rather than manual joining: any value containing a comma — a ruler string,
    # a language name, a failure message — silently shifts every subsequent column when you
    # join by hand, and the file still parses, just wrongly.
    import csv
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(cols)
        for r in rows:
            w.writerow(["" if r.get(c) is None else r.get(c) for c in cols])


def _fmt(x, p=3):
    return "-" if x is None else f"{x:.{p}f}"


def _write_report(path, traj, per_ckpt, lr, manifest):
    L = ["# Phoneme-Adherence Evaluation Report", ""]
    L += [f"- **Ruler:** `{manifest['ruler']}`",
          f"- **Val set:** `{manifest['val_jsonl']}` ({manifest['n_val_rows']} rows)",
          f"- **Generated:** {manifest['generated_utc']}",
          f"- **Adherence samples/lang:** {manifest['adherence_samples_per_lang']}  "
          f"**Probe sentences/lang:** {manifest['probe_sentences_per_lang']}"]
    if manifest.get("budget_scale"):
        L.append(f"- **Budget rescale ACTIVE** (phoneme budget converted to the character "
                 f"budget the model was taught): `{manifest['budget_scale']}`")
    if not manifest["chrf_available"]:
        L.append(f"- **chrF++ UNAVAILABLE** — `{manifest['chrf_unavailable_reason']}`. "
                 f"The fidelity column is absent, not zero.")
    if not manifest["semantic_available"]:
        L.append("- **Semantic scoring UNAVAILABLE** — embedder failed to load. "
                 "Semantic columns are absent, not zero.")
    L += ["", "> Read the per-language table first. Eleven languages across two families do "
          "not plateau together; every aggregate number below hides that.", ""]

    # --- per-language first, by design ---
    L += ["## Per-language", ""]
    by_ckpt = defaultdict(list)
    for r in per_ckpt:
        by_ckpt[(r["step"], r["checkpoint"])].append(r)
    for (step, ckpt) in sorted(by_ckpt):
        L += [f"### {ckpt} (step {step})", "",
              "| Lang | relErr | signed | probe slope | pop slope | chrF++ | semantic | degraded |",
              "|---|---|---|---|---|---|---|---|"]
        for r in sorted(by_ckpt[(step, ckpt)], key=lambda x: x.get("language") or ""):
            L.append(
                f"| {r.get('language')} | {_fmt(r.get('adherence_rel_mean'))} | "
                f"{_fmt(r.get('adherence_signed_mean'))} | {_fmt(r.get('length_slope_probe'))} | "
                f"{_fmt(r.get('length_slope_population'))} | {_fmt(r.get('chrf_mean'), 1)} | "
                f"{_fmt(r.get('semantic_mean'))} | {_fmt(r.get('semantic_degraded_frac'))} |")
        L.append("")

    # --- aggregate ---
    L += ["## Aggregate (read second)", "",
          "| Checkpoint | Step | CE | PPL | relErr | signed | chrF++ | semantic | probe slope | pop slope |",
          "|---|---|---|---|---|---|---|---|---|---|"]
    for t in sorted(traj, key=lambda x: x["step"]):
        L.append(f"| {t['checkpoint']} | {t['step']} | {_fmt(t['ce_mean'], 4)} | "
                 f"{_fmt(t['ce_perplexity'])} | {_fmt(t['adherence_rel_mean'])} | "
                 f"{_fmt(t['adherence_signed_mean'])} | {_fmt(t['chrf_mean'], 1)} | "
                 f"{_fmt(t.get('semantic_mean'))} | {_fmt(t.get('length_slope_probe'))} | "
                 f"{_fmt(t['length_slope_population'])} |")

    v = stopping_verdict(traj)
    L += ["", "## Stopping verdict", "", f"**{v['verdict']}** — {v.get('reason', '')}", "",
          "> `length_slope_probe` holds the sentence fixed and sweeps only the budget: it is "
          "the capability measurement. `length_slope_population` regresses across different "
          "sentences and is confounded by sentence length — a model that ignores the budget "
          "entirely still scores high on it. Never select a checkpoint on the population slope.", ""]

    Path(path).write_text("\n".join(L) + "\n", encoding="utf-8")


def _cli():
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--val_jsonl", required=True)
    p.add_argument("--checkpoints", nargs="*", default=[])
    p.add_argument("--checkpoints_glob", default=None)
    p.add_argument("--base_baseline", action="store_true")
    p.add_argument("--base_model_id", default="meta-llama/Llama-3.1-8B-Instruct")
    p.add_argument("--output_dir", required=True)
    p.add_argument("--mode", choices=["all", "ce", "adherence", "length"], default="all")
    p.add_argument("--max_seq_length", type=int, default=512)
    p.add_argument("--adherence_samples_per_lang", type=int, default=40)
    p.add_argument("--ce_samples_per_lang", type=int, default=150)
    p.add_argument("--probe_sentences_per_lang", type=int, default=30,
                   help="Sentences per language for the capability probe; each costs 5 "
                        "generations. The previous default of 10 gave 50 points per "
                        "language, at which per-language orderings are not trustworthy.")
    p.add_argument("--batch_size", type=int, default=8,
                   help="Generation batch size. The probe is generation-bound; batching is "
                        "what makes a trustworthy sentence count affordable.")
    p.add_argument("--budget_scale_json", default=None,
                   help="Path to a tools/ruler_audit.py report (or a plain {lang: k} map). "
                        "Converts the true-phoneme budget into the character budget a "
                        "character-ruled checkpoint was actually taught — the salvage path.")
    p.add_argument("--semantic_threshold", type=float, default=0.80)
    p.add_argument("--semantic_device", default=None)
    p.add_argument("--no_semantic", action="store_true",
                   help="Skip semantic scoring (faster; loses the production-side fidelity axis).")
    p.add_argument("--wandb", action="store_true", help="Mirror the report into W&B.")
    p.add_argument("--wandb_project", default="indic-dubbing-v3")
    # The personal namespace `nktthegreat` holds ZERO projects; everything lives
    # under the team entity. Getting this wrong sends metrics to a namespace nobody
    # looks at, and the run appears to have logged nothing.
    p.add_argument("--wandb_entity", default="nktthegreat-soccernet")
    p.add_argument("--wandb_run_name", default=None)
    p.add_argument("--wandb_required", action="store_true",
                   help="Abort before loading a model if W&B cannot start. Pass this for "
                        "any Kaggle session: session 02i burned ~12 GPU-hours with no live "
                        "channel because a secrets-service outage was only a warning.")
    p.add_argument("--dump_samples_langs", nargs="*", default=[])
    p.add_argument("--dump_samples_n", type=int, default=8)
    run(p.parse_args())


if __name__ == "__main__":
    _cli()


In [ ]:
%%writefile /kaggle/working/pipeline_v3/evaluation/response_diagnosis.py
"""
evaluation/response_diagnosis.py
=================================
Turns a raw budget-response sweep into a **pre-registered** diagnosis and action.

WHY PRE-REGISTERED
------------------
"Slope 0.35" is compatible with at least four different defects that need four different
fixes. Deciding which one you are looking at *after* seeing the numbers, with a quota
half-spent and a deadline in view, is how a project talks itself into the cheap answer.
So the thresholds and the actions they imply are written here, before the run, and the
classifier is applied mechanically to whatever comes back.

THE FOUR THINGS A LOW SLOPE CAN MEAN
-------------------------------------
Obeying a phoneme budget is not one skill. It is three, and they fail differently:

  (a) NOTICING the number at all
  (b) ESTIMATING how long your own candidate output will be
  (c) REWRITING at a target length without wrecking the meaning

  FLAT        output length barely moves whatever you ask for      -> (a) fails
  NOISY       length moves, but not with the request               -> (b) fails
  SATURATING  tracks near the natural length, refuses to go far    -> (c) fails
  ASYMMETRIC  one direction works, the other does not              -> (c) fails, one way

Each has a different fix, and three of the four are NOT fixed by more training on the same
data. Distinguishing them is the entire purpose of the sweep.

NORMALISATION, AND WHY THE OLD PROBE SLOPE HAD A FLOOR
------------------------------------------------------
Points are normalised per sentence to `produced / natural` against `scale`, and fitted
**with an intercept**. Both choices are load-bearing, and getting them wrong is what made
the previous numbers hard to read.

The old probe regressed raw `produced` on raw `requested`, pooling points from sentences of
different lengths. But `requested = scale x natural`, so the between-sentence spread of
`natural` leaks straight into the fit. Consequence: a model that emits its natural
translation every time, completely ignoring the budget, does **not** score 0. Measured on
this project's actual validation sentences it scores **0.60 to 0.83**, per language:

    lang  as    bn    gu    hi    kn    ml    mr    or    pa    ta    te
    floor .788  .747  .603  .832  .707  .748  .787  .741  .831  .651  .766

So a raw probe slope is not interpretable without its floor, and the floor is different in
every language. Six of the eleven scored at or below their own floor at checkpoint 3801 —
that is the real meaning of "the budget is being ignored".

Fitting `ratio = a + k*scale` per sentence removes both problems. `natural` divides out, so
sentence length cannot leak in, and the intercept absorbs the constant part — leaving
k = 0 for a model that ignores the budget and k = 1 for one that follows it, in every
language, with no floor to subtract.
"""

from __future__ import annotations

import collections
import statistics as st
from typing import Iterable, Optional

# ---------------------------------------------------------------------------------------
# Pre-registered thresholds. Set before the run. Do not tune these to make a result
# come out a particular way — if they are wrong, change them and re-diagnose everything.
# ---------------------------------------------------------------------------------------
OBEYS_SLOPE = 0.70          # at or above this, with a decent fit, the budget is followed
OBEYS_R2 = 0.60
FLAT_SLOPE = 0.30           # below this, and barely varying, the number is not being read
FLAT_CV = 0.12              # coefficient of variation of produced/natural within a sentence
ASYMMETRY = 0.35            # gap between the compress-side and expand-side slopes
SATURATE_MID = 0.60         # responds near 1.0x ...
SATURATE_FAR = 0.30         # ... but not out at the extremes
NOISY_R2 = 0.35

MID_BAND = (0.85, 1.15)
FAR_LOW = 0.70              # scales at or below this are the "far compression" band
FAR_HIGH = 1.60             # scales at or above this are the "far expansion" band

# What each diagnosis implies. These are decisions, made in advance, in one place.
ACTIONS = {
    "OBEYS": (
        "No fix needed. Hold this language out of any corrective training so it cannot "
        "regress, and keep it in the eval set as a forgetting canary."),
    "FLAT": (
        "The budget is not being read. This is the degenerate-objective signature and it "
        "is fixed by DATA, not by more steps: build elastic rows (same English, several "
        "budgets) until ignoring the number stops being a correct answer. Expect a large "
        "gain; this is the cheapest failure mode to fix."),
    "NOISY": (
        "The model cannot estimate its own output length. More elastic rows will NOT fix "
        "this on their own — the supervision it needs is length feedback, not more "
        "examples. Options, in order of cost: (1) rejection-sample its own generations "
        "and train only on the ones that verifiably landed, which is length feedback in "
        "SFT clothing; (2) an auxiliary length-prediction head; (3) fall back to the "
        "deployed v2 generate-and-select path for this language, which already works."),
    "SATURATING": (
        "The budget is read and followed near the natural length, then refuses. Check "
        "whether the floor coincides with semantic collapse: if meaning is intact at the "
        "floor, this is a learned length prior and needs elastic rows AT the extremes, "
        "not more rows near 1.0x. If meaning is already breaking there, the floor is real "
        "and the honest fix is to cap what the pipeline asks of this language."),
    "ASYMMETRIC": (
        "One direction is trained and the other is not. Rebuild augmentation weighted "
        "toward the missing direction. Note the existing augmenter was 69% expand, and "
        "dubbing needs compression."),
    "WEAK": (
        "Partial response with no clean signature. Do not guess: widen the sweep and "
        "raise sentences-per-language for this one before committing training time."),
}


def _fit(xs: list[float], ys: list[float]) -> tuple[Optional[float], Optional[float]]:
    """OLS y = a + kx, returning (k, R^2).

    WITH an intercept, deliberately. Through the origin, a model that emits a constant
    ratio of 1.0 at every scale scores k = mean(s)/mean(s^2) ~ 0.82 on this sweep — it
    would look like it was following the budget. The intercept is what makes k = 0 mean
    "ignores the budget".
    """
    n = len(xs)
    if n < 3:
        return None, None
    mx, my = st.fmean(xs), st.fmean(ys)
    sxx = sum((x - mx) ** 2 for x in xs)
    if sxx == 0:
        return None, None
    sxy = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    syy = sum((y - my) ** 2 for y in ys)
    k = sxy / sxx
    r2 = (sxy * sxy) / (sxx * syy) if syy > 0 else None
    return k, r2


def _slope(points: list[tuple[float, float]]) -> tuple[Optional[float], Optional[float]]:
    if len(points) < 3:
        return None, None
    return _fit([p[0] for p in points], [p[1] for p in points])


def diagnose_language(rows: Iterable[dict]) -> dict:
    """`rows` are raw sweep points for ONE language:
    {sentence_idx, scale, requested_n, produced_n, natural_n, semantic?}
    """
    pts: list[tuple[float, float]] = []          # (scale, produced/natural)
    by_sentence: dict[int, list[tuple[float, float]]] = collections.defaultdict(list)
    sem_by_band: dict[str, list[float]] = collections.defaultdict(list)

    for r in rows:
        nat = float(r.get("natural_n") or 0)
        prod = float(r.get("produced_n") or 0)
        scale = float(r.get("scale") or 0)
        if nat <= 0 or prod <= 0 or scale <= 0:
            continue
        ratio = prod / nat
        pts.append((scale, ratio))
        by_sentence[int(r.get("sentence_idx", -1))].append((scale, ratio))
        s = r.get("semantic")
        if s is not None:
            band = "far_low" if scale <= FAR_LOW else ("far_high" if scale >= FAR_HIGH else "mid")
            sem_by_band[band].append(float(s))

    if len(pts) < 6:
        return {"diagnosis": "WEAK", "n_points": len(pts),
                "note": "too few usable points to classify"}

    slope, r2 = _slope(pts)
    slope_comp, _ = _slope([p for p in pts if p[0] <= 1.0])
    slope_exp, _ = _slope([p for p in pts if p[0] >= 1.0])
    slope_mid, _ = _slope([p for p in pts if MID_BAND[0] <= p[0] <= MID_BAND[1]])
    slope_far_low, _ = _slope([p for p in pts if p[0] <= FAR_LOW])
    slope_far_high, _ = _slope([p for p in pts if p[0] >= FAR_HIGH])

    # Does the output length vary at all WITHIN a sentence as the budget moves? This is the
    # question a pooled slope cannot answer: a model emitting one fixed length per sentence
    # still gets a non-zero pooled slope because longer sentences have longer fixed lengths.
    cvs = []
    for _, sp in by_sentence.items():
        ratios = [r for _, r in sp]
        if len(ratios) >= 4 and st.fmean(ratios) > 0:
            cvs.append(st.pstdev(ratios) / st.fmean(ratios))
    cv = st.fmean(cvs) if cvs else None

    out = {
        "n_points": len(pts), "n_sentences": len(by_sentence),
        "slope": slope, "r2": r2,
        "slope_compress": slope_comp, "slope_expand": slope_exp,
        "slope_mid": slope_mid,
        "slope_far_compress": slope_far_low, "slope_far_expand": slope_far_high,
        "within_sentence_cv": cv,
        "semantic_mid": st.fmean(sem_by_band["mid"]) if sem_by_band.get("mid") else None,
        "semantic_far_compress": (st.fmean(sem_by_band["far_low"])
                                  if sem_by_band.get("far_low") else None),
    }

    asym = (abs(slope_comp - slope_exp)
            if slope_comp is not None and slope_exp is not None else None)
    out["asymmetry"] = asym

    # Priority order matters: OBEYS first so a good language is never pathologised, FLAT
    # second because it is the one signature that is unambiguous.
    if slope is not None and r2 is not None and slope >= OBEYS_SLOPE and r2 >= OBEYS_R2:
        d = "OBEYS"
    elif cv is not None and cv < FLAT_CV and (slope or 0) < FLAT_SLOPE:
        d = "FLAT"
    elif asym is not None and asym >= ASYMMETRY:
        d = "ASYMMETRIC"
    elif (slope_mid is not None and slope_mid >= SATURATE_MID
          and min([s for s in (slope_far_low, slope_far_high) if s is not None] or [1.0])
          < SATURATE_FAR):
        d = "SATURATING"
    elif r2 is not None and r2 < NOISY_R2:
        d = "NOISY"
    else:
        d = "WEAK"

    out["diagnosis"] = d
    out["action"] = ACTIONS[d]
    return out


def diagnose(points: Iterable[dict]) -> dict:
    """All languages. `points` is the flat sweep table."""
    by_lang: dict[str, list[dict]] = collections.defaultdict(list)
    for r in points:
        by_lang[r.get("language")].append(r)
    per_lang = {lg: diagnose_language(rs) for lg, rs in sorted(by_lang.items()) if lg}

    counts = collections.Counter(v["diagnosis"] for v in per_lang.values())
    # The corpus-level call is the majority failure mode among the languages that are not
    # already fine, because that is what the next training run has to be built for.
    broken = [d for d in (v["diagnosis"] for v in per_lang.values()) if d != "OBEYS"]
    dominant = collections.Counter(broken).most_common(1)[0][0] if broken else "OBEYS"
    return {
        "per_language": per_lang,
        "counts": dict(counts),
        "dominant_failure": dominant,
        "corpus_action": ACTIONS[dominant],
        "obeying": sorted(lg for lg, v in per_lang.items() if v["diagnosis"] == "OBEYS"),
    }


def format_report(result: dict) -> str:
    lines = ["", "Budget-response diagnosis", "=" * 108,
             f"{'lang':<6}{'diagnosis':<13}{'slope':>8}{'R2':>7}{'compress':>10}"
             f"{'expand':>9}{'mid':>8}{'farComp':>9}{'within-CV':>11}{'sem@0.7x':>11}"]
    lines.append("-" * 108)
    f = lambda v: f"{v:>8.3f}" if isinstance(v, (int, float)) else f"{'-':>8}"
    for lg, v in result["per_language"].items():
        lines.append(
            f"{lg:<6}{v['diagnosis']:<13}{f(v.get('slope'))}"
            f"{f(v.get('r2'))[1:]}{f(v.get('slope_compress'))[:10]:>10}"
            f"{f(v.get('slope_expand'))[:9]:>9}{f(v.get('slope_mid'))[:8]:>8}"
            f"{f(v.get('slope_far_compress'))[:9]:>9}"
            f"{(f'{v['within_sentence_cv']:.3f}' if v.get('within_sentence_cv') is not None else '-'):>11}"
            f"{(f'{v['semantic_far_compress']:.3f}' if v.get('semantic_far_compress') is not None else '-'):>11}")
    lines += ["", f"counts: {result['counts']}",
              f"already obeying: {result['obeying'] or 'none'}",
              f"dominant failure: {result['dominant_failure']}", "",
              "ACTION", "-" * 108]
    lines += ["  " + l for l in result["corpus_action"].split(". ")]
    return "\n".join(lines)


In [ ]:
%%writefile /kaggle/working/pipeline_v3/evaluation/budget_sweep.py
"""
evaluation/budget_sweep.py
===========================
Step 1 of the Phase-02 repair: find out **why** five languages do not follow the phoneme
budget, and come away with usable training data either way.

THE QUESTION
------------
At checkpoint 3801, six languages follow the budget and five do not. "Slope 0.35" is
compatible with at least four different defects that need four different fixes (see
`evaluation/response_diagnosis.py`). You cannot tell them apart from a slope. You can tell
them apart instantly from the SHAPE of the response curve, which nobody has ever looked at
because only the fitted slope was ever saved.

So: hold the sentence fixed, sweep the budget across a deliberately wide range, and keep
every raw point.

WHY THE SWEEP IS WIDER THAN BEFORE
-----------------------------------
The old probe swept 0.6x to 1.4x. Two problems. First, saturation outside that band is
invisible — a model that tracks to 0.8x and then refuses looks identical to one that
follows everywhere. Second, and worse, the old estimator was not zero-referenced: pooling
raw lengths across sentences let sentence-length variance leak into the slope, so a model
that ignores the budget entirely scored 0.60-0.83 depending on the language rather than 0.
This module reports the within-sentence normalised slope, where 0 means ignored.

WHY IT ALSO HARVESTS
--------------------
A GPU session that produces only a diagnosis is a session that has to be followed by
another session before anything improves. This one generates thousands of translations at
off-natural budgets, and every one of them is exactly measurable — the phoneme counter is
exact and the semantic gate is the same one production uses. So any generation that
verifiably landed near its budget AND kept its meaning is a valid training row for the
corrective fine-tune, harvested for free from work that had to happen anyway.

That is the general rule this file exists to follow:

    Every GPU run must leave behind an artifact the next run consumes.
    A run whose only output is knowledge is a run you will have to pay for twice.

ORDER OF WORK, AND THE WALL CLOCK
----------------------------------
Decision-critical first, opportunistic second, hard stop always. The sweep runs first and
writes after every language, so an interrupted session still answers the question. The
harvest runs only with time left over. `--time_budget_s` is enforced between languages, so
a session cannot overrun its quota allocation.

USAGE
-----
    python -m evaluation.budget_sweep \
        --val_jsonl data/val.phonemes.jsonl \
        --checkpoint checkpoints/checkpoint-3801 \
        --output_dir sweep_out --time_budget_s 5400 --wandb --wandb_required
"""

from __future__ import annotations

import argparse
import csv
import json
import logging
import os
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from common.languages import LANGUAGES, get_language  # noqa: E402
from common.phonemes import assert_g2p_available, count_phonemes, ruler_id  # noqa: E402
from evaluation.response_diagnosis import diagnose, format_report  # noqa: E402

logger = logging.getLogger("budget_sweep")

# Wide on purpose. 0.4x and 2.0x are outside anything the product would ask for, and that
# is the point: the extremes are where saturation shows up, and a curve is only readable
# if it has been driven past its limits.
DEFAULT_SCALES = [0.4, 0.55, 0.7, 0.85, 1.0, 1.15, 1.3, 1.6, 2.0]

# What counts as a harvested training row. Both gates, or neither.
HARVEST_REL_TOL = 0.10          # landed within 10% of the requested budget
HARVEST_SEM_MIN = 0.80          # and did not get there by deleting the meaning
HARVEST_SCALES = [0.6, 0.75, 0.9]   # compression: what dubbing actually asks for
HARVEST_SAMPLES = 4
HARVEST_TEMPERATURE = 0.9       # higher than the sweep — we want spread to select from

PROMPT = '[Translate to {language}] [Target Phonemes: {n}] "{english}"'


def _write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    cols, seen = [], set()
    for r in rows:
        for k in r:
            if k not in seen:
                seen.add(k)
                cols.append(k)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(cols)
        for r in rows:
            w.writerow(["" if r.get(c) is None else r.get(c) for c in cols])


def load_rows(path: str) -> dict[str, list[dict]]:
    by_lang: dict[str, list[dict]] = defaultdict(list)
    rulers = set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            if r.get("language") in LANGUAGES:
                by_lang[r["language"]].append(r)
            rulers.add(r.get("ruler", "MISSING"))
    if not any(str(x).startswith("phonemes:") for x in rulers):
        raise SystemExit(
            f"{path} is not phoneme-ruled (rulers={rulers}). Measuring a phoneme budget "
            f"against character labels is the defect this whole phase exists to undo.")
    logger.info("%d languages | rulers: %s", len(by_lang), rulers)
    return by_lang


# ========================================================================================

def sweep_language(model, tok, lang: str, rows: list[dict], scales: list[float],
                   n_sentences: int, generate_batch, semantic, batch_size: int,
                   point_sink: list, cand_sink: list) -> None:
    """One language, every sentence at every scale. Appends raw points and candidates."""
    lang_name = get_language(lang).name
    specs, naturals, metas = [], [], []
    for r in rows[:n_sentences]:
        eng = r.get("english") or ""
        ref = r.get("target") or r.get("completion") or ""
        if not eng or not ref:
            continue
        try:
            natural = count_phonemes(ref, lang)
        except Exception:  # noqa: BLE001
            continue
        if not natural:
            continue
        si = len(naturals)
        naturals.append(natural)
        metas.append({"english": eng, "reference": ref})
        for s in scales:
            want = max(1, round(natural * s))
            specs.append((si, s, want, PROMPT.format(language=lang_name, n=want, english=eng)))

    if not specs:
        logger.warning("%s: no usable sentences", lang)
        return

    gens = generate_batch(model, tok, [s[3] for s in specs],
                          temperature=0.3, batch_size=batch_size)

    # The semantic anchor is the model's own 1.0x generation, because at dub time there is
    # no reference to compare against — the shipped gate has to work without one.
    anchor: dict[int, str] = {}
    for (si, s, _, _), gen in zip(specs, gens):
        if s == 1.0 and gen:
            anchor[si] = gen

    for (si, s, want, _), gen in zip(specs, gens):
        if not gen:
            continue
        try:
            produced = count_phonemes(gen, lang)
        except Exception:  # noqa: BLE001
            continue
        if not produced:
            continue
        sem_ref = semantic.similarity(metas[si]["reference"], gen) if semantic else None
        sem_anchor = (semantic.similarity(anchor[si], gen)
                      if semantic and si in anchor else None)
        point_sink.append({
            "language": lang, "sentence_idx": si, "scale": s,
            "natural_n": naturals[si], "requested_n": want, "produced_n": produced,
            "semantic": sem_ref, "semantic_vs_own_1x": sem_anchor,
        })
        rel = abs(produced - want) / want
        cand_sink.append({
            "language": lang, "english": metas[si]["english"],
            "reference": metas[si]["reference"], "scale": s, "natural_n": naturals[si],
            "requested_n": want, "produced_n": produced, "rel_err": round(rel, 4),
            "semantic": sem_ref, "generated": gen, "source": "sweep",
            "harvestable": bool(rel <= HARVEST_REL_TOL
                                and (sem_ref is None or sem_ref >= HARVEST_SEM_MIN)),
        })


def harvest_language(model, tok, lang: str, rows: list[dict], n_sentences: int,
                     generate_batch, semantic, batch_size: int, cand_sink: list) -> int:
    """Best-of-n at compression budgets, kept only when both gates pass.

    This is rejection sampling with an EXACT verifier. There is no reward model and no
    judge: the phoneme counter says precisely whether the candidate hit its budget, and the
    semantic gate says whether it cheated to get there. That combination is rare enough to
    be worth exploiting — it turns "the model can sometimes do this" into supervised data
    for "the model should always do this".
    """
    lang_name = get_language(lang).name
    specs, meta = [], []
    for r in rows[:n_sentences]:
        eng = r.get("english") or ""
        ref = r.get("target") or r.get("completion") or ""
        if not eng or not ref:
            continue
        try:
            natural = count_phonemes(ref, lang)
        except Exception:  # noqa: BLE001
            continue
        if not natural:
            continue
        for s in HARVEST_SCALES:
            want = max(1, round(natural * s))
            for _ in range(HARVEST_SAMPLES):
                specs.append(PROMPT.format(language=lang_name, n=want, english=eng))
                meta.append((eng, ref, natural, s, want))

    if not specs:
        return 0
    gens = generate_batch(model, tok, specs, temperature=HARVEST_TEMPERATURE,
                          batch_size=batch_size)

    # Keep the single best candidate per (sentence, budget) rather than every passing one:
    # four near-duplicates of the same sentence would let one easy example dominate.
    best: dict[tuple, dict] = {}
    for (eng, ref, natural, s, want), gen in zip(meta, gens):
        if not gen:
            continue
        try:
            produced = count_phonemes(gen, lang)
        except Exception:  # noqa: BLE001
            continue
        if not produced:
            continue
        rel = abs(produced - want) / want
        if rel > HARVEST_REL_TOL:
            continue
        sem = semantic.similarity(ref, gen) if semantic else None
        if sem is not None and sem < HARVEST_SEM_MIN:
            continue
        key = (eng, s)
        prev = best.get(key)
        if prev is None or rel < prev["rel_err"]:
            best[key] = {
                "language": lang, "english": eng, "reference": ref, "scale": s,
                "natural_n": natural, "requested_n": want, "produced_n": produced,
                "rel_err": round(rel, 4), "semantic": sem, "generated": gen,
                "source": "harvest", "harvestable": True,
            }
    cand_sink.extend(best.values())
    return len(best)


# ========================================================================================

def run(args) -> int:
    logging.basicConfig(level=logging.INFO,
                        format="%(asctime)s %(levelname)s %(message)s")
    t0 = time.time()
    g2p = assert_g2p_available()
    logger.info("G2P preflight PASSED — ruler=%s", ruler_id())

    # Imported here, not at module scope: this file must stay importable on a machine with
    # no torch so the diagnosis can be re-run offline against a saved sweep.
    from evaluation.phoneme_adherence_eval import (  # noqa: PLC0415
        SemanticScorer, generate_batch, load_model, wandb_init,
    )

    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    scales = [float(x) for x in args.scales] if args.scales else DEFAULT_SCALES
    by_lang = load_rows(args.val_jsonl)
    langs = [lg for lg in (args.languages or sorted(by_lang)) if lg in by_lang]

    wb = wandb_init(
        {"job": "budget_sweep", "ruler": g2p["ruler"], "scales": scales,
         "sentences_per_lang": args.sentences_per_lang, "checkpoint": args.checkpoint,
         "time_budget_s": args.time_budget_s},
        args.wandb_project, args.wandb_entity,
        args.wandb_run_name or "02j-budget-sweep",
        required=args.wandb_required) if args.wandb else None

    semantic = None if args.no_semantic else SemanticScorer()
    model, tok = load_model(args.checkpoint, args.base_model_id, args.max_seq_length)

    points: list[dict] = []
    cands: list[dict] = []
    done: list[str] = []

    def flush():
        _write_csv(out_dir / "sweep_points.csv", points)
        with open(out_dir / "candidates.jsonl", "w", encoding="utf-8") as f:
            for c in cands:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")

    # ---- Phase 1: the diagnosis. Decision-critical, so it goes first. -----------------
    for lg in langs:
        if args.time_budget_s and time.time() - t0 > args.time_budget_s:
            logger.warning("time budget reached before %s — stopping the sweep with "
                           "%d/%d languages done", lg, len(done), len(langs))
            break
        t = time.time()
        sweep_language(model, tok, lg, by_lang[lg], scales, args.sentences_per_lang,
                       generate_batch, semantic, args.batch_size, points, cands)
        done.append(lg)
        flush()

        d = diagnose([p for p in points if p["language"] == lg])["per_language"].get(lg, {})
        logger.info("%s: %s  slope=%.3f r2=%.3f  (%.0fs)", lg,
                    d.get("diagnosis", "?"), d.get("slope") or float("nan"),
                    d.get("r2") or float("nan"), time.time() - t)
        if wb is not None:
            try:
                wb.log({f"{lg}/slope_normalized": d.get("slope"),
                        f"{lg}/r2_normalized": d.get("r2"),
                        f"{lg}/slope_compress": d.get("slope_compress"),
                        f"{lg}/slope_expand": d.get("slope_expand"),
                        f"{lg}/within_sentence_cv": d.get("within_sentence_cv"),
                        f"{lg}/semantic_far_compress": d.get("semantic_far_compress"),
                        "languages_done": len(done)})
            except Exception as e:  # noqa: BLE001
                logger.error("wandb log failed for %s: %s", lg, e)

    result = diagnose(points)
    (out_dir / "DIAGNOSIS.json").write_text(
        json.dumps({"generated_utc": datetime.now(timezone.utc).isoformat(),
                    "ruler": g2p["ruler"], "scales": scales,
                    "languages_swept": done, **result}, indent=2, ensure_ascii=False),
        encoding="utf-8")
    print(format_report(result))

    # ---- Phase 2: harvest, only with time to spare. -----------------------------------
    harvested = {}
    if args.harvest:
        weak = [lg for lg in done
                if result["per_language"].get(lg, {}).get("diagnosis") != "OBEYS"]
        for lg in weak:
            left = args.time_budget_s - (time.time() - t0) if args.time_budget_s else 1e9
            if left < args.harvest_reserve_s:
                logger.warning("stopping harvest before %s — %.0fs left, need %.0fs",
                               lg, left, args.harvest_reserve_s)
                break
            n = harvest_language(model, tok, lg, by_lang[lg], args.harvest_sentences,
                                 generate_batch, semantic, args.batch_size, cands)
            harvested[lg] = n
            flush()
            logger.info("%s: harvested %d verified elastic rows", lg, n)
            if wb is not None:
                try:
                    wb.log({f"{lg}/harvested_rows": n})
                except Exception:  # noqa: BLE001
                    pass

    flush()
    n_ok = sum(1 for c in cands if c.get("harvestable"))
    summary = {
        "languages_swept": done, "n_points": len(points),
        "n_candidates": len(cands), "n_harvestable": n_ok,
        "harvested_per_lang": harvested,
        "elapsed_s": round(time.time() - t0),
        "diagnosis_counts": result["counts"],
        "dominant_failure": result["dominant_failure"],
    }
    (out_dir / "SUMMARY.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("\n" + json.dumps(summary, indent=2))

    if wb is not None:
        try:
            wb.summary.update(summary)
            wb.finish()
        except Exception as e:  # noqa: BLE001
            logger.error("wandb finalisation failed: %s", e)
    logger.info("Wrote outputs to %s", out_dir)
    return 0


def main() -> int:
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--val_jsonl", required=True)
    p.add_argument("--checkpoint", default=None, help="Adapter dir; omit for the base model.")
    p.add_argument("--base_model_id", default="meta-llama/Llama-3.1-8B-Instruct")
    p.add_argument("--output_dir", required=True)
    p.add_argument("--max_seq_length", type=int, default=512)
    p.add_argument("--batch_size", type=int, default=24)
    p.add_argument("--languages", nargs="*", default=None)
    p.add_argument("--scales", nargs="*", default=None)
    p.add_argument("--sentences_per_lang", type=int, default=40)
    p.add_argument("--no_semantic", action="store_true")
    p.add_argument("--time_budget_s", type=int, default=0,
                   help="Hard wall-clock stop, enforced between languages. 0 = no limit.")
    p.add_argument("--harvest", action="store_true",
                   help="After the sweep, best-of-n at compression budgets to build "
                        "verified elastic training rows.")
    p.add_argument("--harvest_sentences", type=int, default=60)
    p.add_argument("--harvest_reserve_s", type=int, default=420,
                   help="Do not start another language's harvest with less than this left.")
    p.add_argument("--wandb", action="store_true")
    p.add_argument("--wandb_required", action="store_true")
    p.add_argument("--wandb_project", default="indic-dubbing-v3")
    p.add_argument("--wandb_entity", default="nktthegreat-soccernet")
    p.add_argument("--wandb_run_name", default=None)
    return run(p.parse_args())


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/pipeline_v3/tools/__init__.py


In [ ]:
%%writefile /kaggle/working/pipeline_v3/tools/preflight.py
"""
tools/preflight.py
==================
The gate every GPU session must pass **before it is pushed**.

WHY THIS EXISTS
---------------
Phase 02 spent two weeks and most of a GPU quota on runs whose only product was the
discovery that an earlier run had been invalid. Look at what each discovery actually
required to find:

| discovery | how it was found | what it needed |
|---|---|---|
| corpus labelled in characters, not phonemes | recount the labels, compare | CPU, seconds |
| `train.jsonl` uniformly char-ruled, not mixed | count `frac_label_eq_chars` | CPU, seconds |
| 0.2% of rows carry an off-natural budget | group by (language, english) | CPU, seconds |
| augmentation missing `or`, `pa`, `te` entirely | group by language | CPU, seconds |
| augmentation 69% expand / 31% compress | group by direction | CPU, seconds |
| chrF++ silently returning `None` for 440 rows | import at module scope | CPU, instantly |
| the 11 languages split into two populations | generate and measure | **GPU, hours** |

Exactly one of those needed a GPU. Every other one was a property of a file on disk that
nobody had asked the file about — and each of them invalidated a GPU run already paid for.

So the rule this module enforces is not "test more". It is:

    A GPU session may only be spent on questions that CANNOT be answered on CPU.
    Everything else is a precondition, and preconditions are checked before you pay.

TWO STAGES, BECAUSE THE CHECKS HAVE DIFFERENT HOMES
----------------------------------------------------
`--stage local` runs on the dev machine before `--push`. It needs no espeak and no GPU:
every check is structural, answerable by grouping and counting.

`--stage kaggle` runs inside the notebook, in the first cell, before any model loads. It
adds the checks that need espeak (which is installed there and not here) and aborts the
session in seconds rather than discovering the problem after four hours of generation.

A CHECK THAT CANNOT RUN IS NOT A CHECK THAT PASSED
---------------------------------------------------
Results are three-state: PASS, FAIL, UNKNOWN. UNKNOWN blocks by default, because "I could
not verify this" and "this is fine" are the same outcome only to a process that is trying
to talk itself into launching. `--allow_unknown` exists for the case where you have
genuinely decided to proceed unverified, and it makes that decision explicit and logged.

USAGE
-----
    python -m tools.preflight --stage local  --corpus data/train.jsonl --val data/val.jsonl
    python -m tools.preflight --stage kaggle --corpus /kaggle/working/data/train.jsonl
    python -m tools.preflight --stage local  --corpus ... --json report.json

Exit 0 = safe to spend GPU. Non-zero = do not push.
"""

from __future__ import annotations

import argparse
import collections
import json
import logging
import os
import re
import statistics as st
import sys
from pathlib import Path
from typing import Optional

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from common.languages import LANGUAGES  # noqa: E402

logger = logging.getLogger("preflight")

PROMPT_N = re.compile(r"\[Target Phonemes:\s*(\d+)\]")

# ---------------------------------------------------------------------------------------
# Pre-registered thresholds. They live in code so they are decided while nothing is at
# stake, rather than at 2am with a session already queued and a deadline in view.
# ---------------------------------------------------------------------------------------
MIN_ELASTIC_ROW_FRAC = 0.15     # rows sitting in a group that spans >1 budget
MIN_ELASTIC_SPREAD = 1.25       # median max/min budget within those groups
MIN_COMPRESS_SHARE = 0.40       # of off-base rows, how many ask for LESS
MIN_ROWS_PER_LANG = 200
MIN_ELASTIC_GROUPS_PER_LANG = 50

PASS, FAIL, UNKNOWN = "PASS", "FAIL", "UNKNOWN"


class Check:
    def __init__(self, name: str, state: str, detail: str,
                 data: Optional[dict] = None, blocking: bool = True):
        self.name, self.state, self.detail = name, state, detail
        self.data, self.blocking = data or {}, blocking

    @property
    def ok(self) -> bool:
        return self.state == PASS

    def line(self) -> str:
        mark = {PASS: "PASS", FAIL: "FAIL", UNKNOWN: "????"}[self.state]
        if not self.blocking and self.state != PASS:
            mark = "warn"
        return f"  [{mark}] {self.name:<12} {self.detail}"


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


# ========================================================================================
# Structural checks — no espeak, no GPU. These run on the dev machine.
# ========================================================================================

def _groups(rows: list[dict]) -> dict[tuple, list[int]]:
    """(language, english) -> the budgets that input is seen with."""
    g = collections.defaultdict(list)
    for r in rows:
        try:
            g[(r.get("language"), r.get("english"))].append(int(r.get("n_phonemes", -1)))
        except (TypeError, ValueError):
            continue
    return g


def check_elasticity(rows: list[dict], label: str) -> Check:
    """THE check Phase 02 did not have.

    A model learns to read the budget only if the budget is not deducible from the input.
    Samanantar gives ONE reference translation per English sentence, so the budget written
    into that row's prompt is that translation's own length — a deterministic function of
    the English. On such a row, "ignore the number, translate naturally" is a fully correct
    answer, and gradient descent has no reason to prefer a model that reads the number.

    The signal exists only where the SAME input appears with DIFFERENT budgets. That is
    what makes the number load-bearing.

    Measured on the corpus that trained checkpoint-3801: 0.2% of rows. Malayalam's probe
    slope then moved 0.419 -> 0.418 across 3,801 steps. That is not slow learning; that is
    a language that was never shown the task.
    """
    g = _groups(rows)
    if not g:
        return Check("elasticity", UNKNOWN, f"{label}: no (language, english) groups found")
    multi = {k: v for k, v in g.items() if len(set(v)) > 1}
    n_rows = sum(len(v) for v in g.values())
    rows_in_multi = sum(len(v) for v in multi.values())
    frac = rows_in_multi / n_rows if n_rows else 0.0

    spread = 0.0
    if multi:
        spreads = sorted(max(v) / min(v) for v in multi.values() if min(v) > 0)
        spread = st.median(spreads) if spreads else 0.0

    ok = frac >= MIN_ELASTIC_ROW_FRAC and spread >= MIN_ELASTIC_SPREAD
    detail = (f"{label}: {frac:.1%} of rows sit in a group spanning >1 budget "
              f"(floor {MIN_ELASTIC_ROW_FRAC:.0%}), median spread {spread:.2f}x "
              f"(floor {MIN_ELASTIC_SPREAD:.2f}x)")
    if not ok:
        detail += ("\n               <-- the budget is deducible from the English on "
                   "almost every row, so ignoring it is never penalised")
    return Check("elasticity", PASS if ok else FAIL, detail,
                 {"elastic_row_frac": frac, "median_spread": spread,
                  "n_multi_groups": len(multi), "n_groups": len(g)})


def check_direction(rows: list[dict], label: str) -> Check:
    """Dubbing mostly needs COMPRESSION. The base model overshoots by 34% (signed +0.344),
    so at dub time the operation asked for is almost always "say this in fewer sounds".
    The augmentation that existed was 737 expand / 327 compress: it trained the easy
    direction and under-trained the one the product needs.

    Within each elastic group the baseline is the SMALLEST budget's sibling... no — the
    baseline is the un-augmented row where one exists, else the group median. Every other
    row is then classified against it.
    """
    by_group = collections.defaultdict(list)
    for r in rows:
        by_group[(r.get("language"), r.get("english"))].append(r)

    comp = exp = 0
    for _, grp in by_group.items():
        if len(grp) < 2:
            continue
        budgets = [int(x.get("n_phonemes", -1)) for x in grp]
        if len(set(budgets)) < 2:
            continue
        base_rows = [x for x in grp if not x.get("augmentation")]
        base = (int(base_rows[0].get("n_phonemes", -1)) if base_rows
                else st.median(budgets))
        for x in grp:
            if base_rows and x in base_rows:
                continue
            n = int(x.get("n_phonemes", -1))
            if n < base * 0.98:
                comp += 1
            elif n > base * 1.02:
                exp += 1

    total = comp + exp
    if not total:
        return Check("direction", FAIL,
                     f"{label}: no off-baseline rows at all — nothing teaches either "
                     f"direction")
    share = comp / total
    ok = share >= MIN_COMPRESS_SHARE
    return Check("direction", PASS if ok else FAIL,
                 f"{label}: {share:.0%} of off-baseline rows ask for LESS "
                 f"({comp} compress / {exp} expand, floor {MIN_COMPRESS_SHARE:.0%})",
                 {"compress_share": share, "n_compress": comp, "n_expand": exp})


def check_coverage(rows: list[dict], label: str) -> Check:
    """All 11, and — for a training corpus — all 11 with elastic groups. The augmentation
    that existed covered 7 languages and omitted `or`, `pa` and `te`: three of the five
    later found stuck. The languages that most needed the lesson were the ones left out.

    A validation set is measured, never trained on, so it needs presence and enough rows to
    estimate from, not elasticity. Applying the training thresholds to it would be the same
    category error as reading `adherence_rel_mean` as the objective.
    """
    is_val = label == "val"
    min_rows = 100 if is_val else MIN_ROWS_PER_LANG

    per_lang = collections.Counter(r.get("language") for r in rows)
    g = _groups(rows)
    elastic_per_lang = collections.Counter(k[0] for k, v in g.items() if len(set(v)) > 1)

    problems = []
    missing = sorted(set(LANGUAGES) - set(per_lang))
    if missing:
        problems.append(f"absent: {missing}")
    thin = sorted(lg for lg in LANGUAGES if 0 < per_lang.get(lg, 0) < min_rows)
    if thin:
        problems.append(f"under {min_rows} rows: {thin}")
    if not is_val:
        no_elastic = sorted(lg for lg in LANGUAGES
                            if elastic_per_lang.get(lg, 0) < MIN_ELASTIC_GROUPS_PER_LANG)
        if no_elastic:
            problems.append(
                f"under {MIN_ELASTIC_GROUPS_PER_LANG} elastic groups: {no_elastic}")

    ok_msg = (f"{label}: all {len(LANGUAGES)} languages present, >={min_rows} rows each"
              + ("" if is_val else " and elastic"))
    return Check("coverage", PASS if not problems else FAIL,
                 ok_msg if not problems else f"{label}: " + "; ".join(problems),
                 {"per_lang": dict(per_lang), "elastic_per_lang": dict(elastic_per_lang)})


def check_prompt(rows: list[dict], label: str) -> Check:
    """The model trains on the prompt; the metric reads the field. A corpus where those
    disagree is inconsistent in a way no metric will ever report."""
    bad = checked = 0
    for r in rows:
        p = r.get("prompt")
        if not p:
            continue
        checked += 1
        m = PROMPT_N.search(p)
        if not m or int(m.group(1)) != int(r.get("n_phonemes", -1)):
            bad += 1
    if not checked:
        return Check("prompt", UNKNOWN, f"{label}: no prompts present")
    return Check("prompt", PASS if bad == 0 else FAIL,
                 f"{label}: {checked - bad}/{checked} prompts agree with n_phonemes",
                 {"checked": checked, "bad": bad})


def check_ruler(rows: list[dict], label: str) -> Check:
    from common.phonemes import RULER_PHONEMES
    rulers = collections.Counter(r.get("ruler", "MISSING") for r in rows)
    if len(rulers) != 1:
        return Check("ruler", FAIL,
                     f"{label}: {len(rulers)} distinct rulers {dict(rulers)} — a mixed "
                     f"corpus teaches two contradictory tasks under one prompt token",
                     {"rulers": dict(rulers)})
    only = next(iter(rulers))
    if not str(only).startswith(RULER_PHONEMES):
        return Check("ruler", FAIL,
                     f"{label}: ruler is {only!r}, not a phoneme ruler — run "
                     f"tools/relabel_dataset.py first", {"ruler": only})
    return Check("ruler", PASS, f"{label}: uniform {only!r} across {len(rows)} rows",
                 {"ruler": only})


# ========================================================================================
# Checks that need espeak — these run inside the Kaggle notebook, before any model loads.
# ========================================================================================

def check_g2p(_rows=None, _label=None) -> Check:
    """Not "is a phonemizer importable" — that passed in the session that mislabelled the
    whole corpus. This asserts the output symbols are demonstrably NOT the input's own
    characters."""
    try:
        from common.phonemes import assert_g2p_available
        info = assert_g2p_available()
        return Check("g2p", PASS, f"ruler={info['ruler']} — output is not the input", info)
    except ImportError as e:
        return Check("g2p", UNKNOWN, f"cannot import the counter: {e}")
    except Exception as e:  # noqa: BLE001
        msg = str(e).splitlines()[0]
        state = UNKNOWN if "not importable" in str(e) else FAIL
        return Check("g2p", state, f"{type(e).__name__}: {msg}")


def check_labels(rows: list[dict], label: str, sample: int = 3000) -> Check:
    """The stated count must reproduce from the row's own completion. This is the check
    that would have caught the character ruler on day one: it does not ask whether the
    phonemizer ran, it asks whether the number in the file is the number the counter
    produces."""
    try:
        from common.phonemes import count_phonemes
    except ImportError as e:
        return Check("labels", UNKNOWN, f"counter unavailable: {e}")

    step = max(1, len(rows) // sample)
    checked = bad = 0
    worst = []
    for r in rows[::step]:
        comp = r.get("completion") or r.get("target")
        lang = r.get("language")
        if not comp or lang not in LANGUAGES:
            continue
        try:
            true_n = count_phonemes(comp, lang)
        except Exception as e:  # noqa: BLE001
            return Check("labels", UNKNOWN,
                         f"{label}: cannot recount ({type(e).__name__}: "
                         f"{str(e).splitlines()[0]})")
        checked += 1
        if true_n != int(r.get("n_phonemes", -1)):
            bad += 1
            if len(worst) < 3:
                worst.append(f"{lang}: says {r.get('n_phonemes')}, counts {true_n}")
    if not checked:
        return Check("labels", UNKNOWN, f"{label}: nothing checkable")
    return Check("labels", PASS if bad == 0 else FAIL,
                 f"{label}: {checked - bad}/{checked} labels reproduce exactly"
                 + (f" — e.g. {'; '.join(worst)}" if worst else ""),
                 {"checked": checked, "bad": bad})


# ========================================================================================
# Environment
# ========================================================================================

def check_wandb(entity: str = "nktthegreat-soccernet") -> Check:
    """W&B is the only channel that reports while a Kaggle session runs, so a session
    launched without it is unobservable until it ends. 02i burned ~12 GPU-hours that way."""
    # `_netrc` on Windows, `.netrc` elsewhere — wandb picks by platform.
    if not (os.environ.get("WANDB_API_KEY") or
            any((Path.home() / n).exists() for n in (".netrc", "_netrc"))):
        return Check("wandb", FAIL,
                     "no WANDB_API_KEY and no ~/.netrc or ~/_netrc — the run would be blind")
    try:
        import wandb
        list(wandb.Api().projects(entity))
        return Check("wandb", PASS, f"{entity} reachable")
    except ImportError:
        return Check("wandb", UNKNOWN, "wandb not installed (pip install wandb)")
    except Exception as e:  # noqa: BLE001
        return Check("wandb", FAIL, f"cannot reach {entity}: {str(e).splitlines()[0]}")


def check_checkpoints(paths: list[str]) -> Check:
    """A checkpoint with weights but no `scheduler.pt` resumes at the wrong learning rate
    and looks perfectly healthy doing it."""
    need = ["adapter_model.safetensors", "adapter_config.json"]
    resume = ["scheduler.pt", "trainer_state.json"]
    problems = []
    for p in paths:
        d = Path(p)
        if not d.is_dir():
            problems.append(f"{d.name}: not a directory")
            continue
        for f in need:
            if not (d / f).exists():
                problems.append(f"{d.name}: missing {f}")
        gone = [f for f in resume if not (d / f).exists()]
        if gone:
            problems.append(f"{d.name}: cannot resume — missing {gone}")
    return Check("checkpoints", PASS if not problems else FAIL,
                 f"{len(paths)} checkpoint(s) complete" if not problems
                 else "; ".join(problems))


# ========================================================================================

STRUCTURAL = {"elasticity": check_elasticity, "direction": check_direction,
              "coverage": check_coverage, "prompt": check_prompt, "ruler": check_ruler}
G2P_DEPENDENT = {"labels": check_labels}

STAGES = {
    # Before pushing. No espeak, no GPU — every one of these is grouping and counting.
    "local": ["elasticity", "direction", "coverage", "prompt"],
    # Inside the notebook, first cell, before a model is loaded.
    "kaggle": ["ruler", "labels", "elasticity", "coverage", "prompt"],
    "all": ["ruler", "labels", "elasticity", "direction", "coverage", "prompt"],
}


def main() -> int:
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--stage", choices=sorted(STAGES), default="local")
    p.add_argument("--corpus", default=None, help="Training corpus JSONL.")
    p.add_argument("--val", default=None, help="Validation corpus JSONL.")
    p.add_argument("--checkpoints", nargs="*", default=[])
    p.add_argument("--require", nargs="*", default=None,
                   help="Override the stage's check list.")
    p.add_argument("--warn_only", nargs="*", default=[],
                   help="Run these but do not block on them.")
    p.add_argument("--allow_unknown", action="store_true",
                   help="Let UNKNOWN pass. Makes 'I could not verify this' an explicit, "
                        "logged decision instead of a silent one.")
    p.add_argument("--no_wandb_check", action="store_true")
    p.add_argument("--json", default=None)
    args = p.parse_args()

    checks: list[Check] = []
    wanted = args.require or STAGES[args.stage]

    print(f"\nPreflight — stage {args.stage!r}")
    print("Environment")
    if args.stage == "kaggle" or "labels" in wanted:
        checks.append(check_g2p())
    if not args.no_wandb_check:
        checks.append(check_wandb())
    if args.checkpoints:
        checks.append(check_checkpoints(args.checkpoints))
    for c in checks:
        print(c.line())

    for label, path in (("train", args.corpus), ("val", args.val)):
        if not path:
            continue
        rows = load_jsonl(Path(path))
        print(f"\n{label}: {path}  ({len(rows)} rows)")
        for name in wanted:
            fn = STRUCTURAL.get(name) or G2P_DEPENDENT.get(name)
            if fn is None:
                print(f"  [....] unknown check {name!r}")
                continue
            c = fn(rows, label)
            # A validation set is measured, never trained on: elasticity and direction are
            # properties the TRAINING corpus must have, not this one.
            if label == "val" and name in ("elasticity", "direction"):
                c.blocking = False
            if name in args.warn_only:
                c.blocking = False
            checks.append(c)
            print(c.line())

    blocking = [c for c in checks if c.blocking and c.state == FAIL]
    unknown = [c for c in checks if c.blocking and c.state == UNKNOWN]
    warned = [c for c in checks if not c.blocking and c.state != PASS]

    print()
    if args.json:
        Path(args.json).write_text(json.dumps(
            {"stage": args.stage, "ok": not blocking and not (unknown and not args.allow_unknown),
             "checks": [{"name": c.name, "state": c.state, "blocking": c.blocking,
                         "detail": c.detail, "data": c.data} for c in checks]},
            indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"report -> {args.json}")

    if warned:
        print(f"{len(warned)} non-blocking:")
        for c in warned:
            print(f"    {c.name}: {c.detail}")

    if unknown and not args.allow_unknown:
        print(f"\nPREFLIGHT BLOCKED — {len(unknown)} check(s) could not run:")
        for c in unknown:
            print(f"    {c.name}: {c.detail}")
        print("\nA check that cannot run is not a check that passed. Fix the tooling, or\n"
              "pass --allow_unknown to record that you decided to proceed unverified.")
        return 2

    if blocking:
        print(f"PREFLIGHT FAILED — {len(blocking)} blocking issue(s):")
        for c in blocking:
            print(f"    {c.name}: {c.detail}")
        print("\nDo NOT push. Every one of these is answerable here, on CPU, for free.\n"
              "A GPU session is for questions that cannot be answered any other way.")
        return 1

    print("PREFLIGHT PASSED — safe to spend GPU.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 3. Preflight — G2P, then the corpus, before a model is loaded

Every check here is answerable in seconds. Discovering any of them after four hours of generation is what this phase has been doing wrong.

In [ ]:
import logging, subprocess, sys
logging.basicConfig(level=logging.INFO, format='%(message)s')
from common.phonemes import assert_g2p_available, ruler_id
info = assert_g2p_available()
print('G2P preflight PASSED — ruler =', ruler_id())


## 4. Locate inputs

In [ ]:
from pathlib import Path

CAND = list(Path('/kaggle/input').rglob('val.phonemes.jsonl'))
assert CAND, 'val.phonemes.jsonl not found in /kaggle/input — attach 02h output'
VAL = CAND[0]
print('val :', VAL)

CK = sorted({p.parent for p in Path('/kaggle/input').rglob('adapter_model.safetensors')},
            key=lambda p: int(''.join(c for c in p.name if c.isdigit()) or 0))
assert CK, 'no checkpoints found in /kaggle/input'
CKPT = str(CK[-1])
print('checkpoint:', CKPT)


In [ ]:
# The corpus gate. Structural checks only — the training corpus is not attached
# here, so this validates what the sweep will actually read.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'tools.preflight', '--stage', 'kaggle',
                '--corpus', str(VAL), '--no_wandb_check',
                '--require', 'ruler', 'labels', 'prompt'], check=True)


## 5. Smoke test — one generation, before committing an hour

A `!` cell swallows exit codes, so this is a real assertion. The first attempt at 02i failed here and reported it four cells later as an unrelated missing file.

In [ ]:
import time, json
from evaluation.phoneme_adherence_eval import load_model, generate_batch, true_phoneme_len, SemanticScorer

t0 = time.time()
rows = [json.loads(l) for l in open(VAL, encoding='utf-8')]
row = next(r for r in rows if r['language'] == 'or')
m, tk = load_model(CKPT, 'meta-llama/Llama-3.1-8B-Instruct', 512)

# Ask for HALF the natural length — the regime the old sweep never entered.
nat = int(row['n_phonemes'])
want = max(1, round(nat * 0.5))
p = f'[Translate to Odia] [Target Phonemes: {want}] \"' + row['english'] + '\"'
gen = generate_batch(m, tk, [p], batch_size=1)[0]
got = true_phoneme_len(gen, 'or')
print(f'natural {nat}  requested {want}  produced {got}')
print('generated:', gen[:120])
assert gen and got, 'generation or phonemization failed'

# Throughput, measured rather than assumed — it sets the sweep size below.
t1 = time.time()
batch = generate_batch(m, tk, [p] * 24, batch_size=24)
rate = 24 / (time.time() - t1)
print(f'SMOKE PASSED in {time.time()-t0:.0f}s — {rate:.1f} generations/sec at batch 24')

del m
import gc, torch
gc.collect(); torch.cuda.empty_cache()


## 6. The sweep

11 languages x 40 sentences x 9 scales. Writes after every language, so an interrupted session still answers the question. `--time_budget_s` is a hard stop: on Kaggle's T4x2 every wall-clock hour costs two GPU-hours, so 5400s here is ~3 GPU-hours.

In [ ]:
import subprocess, sys

cmd = [sys.executable, '-m', 'evaluation.budget_sweep',
       '--val_jsonl', str(VAL),
       '--checkpoint', CKPT,
       '--output_dir', '/kaggle/working/sweep',
       '--sentences_per_lang', '40',
       '--batch_size', '24',
       '--time_budget_s', '5400',
       '--harvest', '--harvest_sentences', '60',
       '--wandb', '--wandb_required',
       '--wandb_project', 'indic-dubbing-v3',
       '--wandb_entity', 'nktthegreat-soccernet',
       '--wandb_run_name', '02j-budget-sweep']
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)


## 7. The diagnosis — read this, not the aggregate

In [ ]:
from pathlib import Path
print(Path('/kaggle/working/sweep/DIAGNOSIS.json').read_text(encoding='utf-8')[:4000])


In [ ]:
import json
from evaluation.response_diagnosis import format_report
d = json.loads(Path('/kaggle/working/sweep/DIAGNOSIS.json').read_text(encoding='utf-8'))
print(format_report(d))
print()
print(Path('/kaggle/working/sweep/SUMMARY.json').read_text(encoding='utf-8'))


## 8. What was harvested

Verified elastic rows: the generation hit its budget within 10% AND stayed above the 0.80 semantic gate. These are training data for the corrective fine-tune, produced by a run whose purpose was diagnosis.

In [ ]:
import json, collections
cands = [json.loads(l) for l in open('/kaggle/working/sweep/candidates.jsonl',
                                     encoding='utf-8')]
ok = [c for c in cands if c['harvestable']]
print(f'{len(ok)} harvestable of {len(cands)} generations')
per = collections.Counter(c['language'] for c in ok)
print('per language:', dict(sorted(per.items())))
comp = sum(1 for c in ok if c['scale'] < 1.0)
print(f'compression rows: {comp} ({comp/max(1,len(ok)):.0%}) — dubbing needs these')
for c in ok[:5]:
    print(f"\n  {c['language']} @ {c['scale']}x  want {c['requested_n']} "
          f"got {c['produced_n']}  sem {c['semantic']}")
    print('   ', c['generated'][:110])


Save Version. Step 2 consumes `sweep/candidates.jsonl` and is chosen by `DIAGNOSIS.json`'s `dominant_failure` — the action for each diagnosis was pre-registered in `evaluation/response_diagnosis.py` before this run, so the decision does not get made under pressure.